In [1]:
import duckdb

print("DuckDB version:", duckdb.__version__)

DuckDB version: 1.5.6


In [3]:
import pandas as pd
import duckdb

raw_path = "NBA_Global_Media_Insights/Data/raw/"

master_audience = pd.read_csv(raw_path + "master_audience.csv")
master_schedule = pd.read_csv(raw_path + "master_schedule.csv")
dataset2 = pd.read_csv(raw_path + "tv_media_blog_viewership_2025_26.csv")

print("master_audience:", master_audience.shape)
print("master_schedule:", master_schedule.shape)
print("dataset2:", dataset2.shape)

master_audience: (292, 14)
master_schedule: (1364, 8)
dataset2: (241, 14)


In [4]:
#Registering Datasets in duckdb
con = duckdb.connect()

con.register("master_audience", master_audience)
con.register("master_schedule", master_schedule)
con.register("dataset2", dataset2)

print(con.sql("SHOW TABLES"))

┌─────────────────┐
│      name       │
│     varchar     │
├─────────────────┤
│ dataset2        │
│ master_audience │
│ master_schedule │
└─────────────────┘



In [5]:
#Inspecting Audience Data
con.sql("""
    DESCRIBE master_audience
""")

┌────────────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│    column_name     │ column_type │  null   │   key   │ default │  extra  │
│      varchar       │   varchar   │ varchar │ varchar │ varchar │ varchar │
├────────────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ date               │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ away_team          │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ home_team          │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ match_key          │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ national_tv        │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ viewership         │ DOUBLE      │ YES     │ NULL    │ NULL    │ NULL    │
│ measurement_type   │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ measurement_source │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ measurement_scope  │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │

In [6]:
con.sql("""
    SELECT
        measurement_type,
        measurement_scope,
        COUNT(*) AS observations,
        ROUND(AVG(viewership), 3) AS avg_viewership_millions,
        ROUND(MIN(viewership), 3) AS min_viewership_millions,
        ROUND(MAX(viewership), 3) AS max_viewership_millions
    FROM master_audience
    GROUP BY
        measurement_type,
        measurement_scope
    ORDER BY
        measurement_type,
        measurement_scope
""")

┌──────────────────┬───────────────────┬──────────────┬─────────────────────────┬─────────────────────────┬─────────────────────────┐
│ measurement_type │ measurement_scope │ observations │ avg_viewership_millions │ min_viewership_millions │ max_viewership_millions │
│     varchar      │      varchar      │    int64     │         double          │         double          │         double          │
├──────────────────┼───────────────────┼──────────────┼─────────────────────────┼─────────────────────────┼─────────────────────────┤
│ average          │ game              │          210 │             1731080.952 │                516000.0 │               6711000.0 │
│ average          │ window            │           34 │             1937764.706 │               1453000.0 │               2554000.0 │
│ combined         │ game              │           21 │             2942857.143 │               1200000.0 │               5900000.0 │
│ combined         │ window            │           20 │       

In [7]:
con.sql("""
    SELECT
        measurement_type,
        measurement_scope,
        COUNT(*) AS observations,
        ROUND(AVG(viewership) / 1000000, 2) AS avg_viewership_millions,
        ROUND(MIN(viewership) / 1000000, 2) AS min_viewership_millions,
        ROUND(MAX(viewership) / 1000000, 2) AS max_viewership_millions
    FROM master_audience
    GROUP BY
        measurement_type,
        measurement_scope
    ORDER BY
        measurement_type,
        measurement_scope
""")

┌──────────────────┬───────────────────┬──────────────┬─────────────────────────┬─────────────────────────┬─────────────────────────┐
│ measurement_type │ measurement_scope │ observations │ avg_viewership_millions │ min_viewership_millions │ max_viewership_millions │
│     varchar      │      varchar      │    int64     │         double          │         double          │         double          │
├──────────────────┼───────────────────┼──────────────┼─────────────────────────┼─────────────────────────┼─────────────────────────┤
│ average          │ game              │          210 │                    1.73 │                    0.52 │                    6.71 │
│ average          │ window            │           34 │                    1.94 │                    1.45 │                    2.55 │
│ combined         │ game              │           21 │                    2.94 │                     1.2 │                     5.9 │
│ combined         │ window            │           20 │       

In [8]:
#NETWORK LEVEL AUDIENCE PERFORMANCE
con.sql("""
    SELECT
        national_tv AS network,
        COUNT(*) AS measurements,
        ROUND(AVG(viewership) / 1000000, 2) AS avg_viewership_millions,
        ROUND(MIN(viewership) / 1000000, 2) AS min_viewership_millions,
        ROUND(MAX(viewership) / 1000000, 2) AS max_viewership_millions
    FROM master_audience
    WHERE measurement_type = 'average'
      AND national_tv IS NOT NULL
    GROUP BY national_tv
    ORDER BY avg_viewership_millions DESC
""")

┌───────────────┬──────────────┬─────────────────────────┬─────────────────────────┬─────────────────────────┐
│    network    │ measurements │ avg_viewership_millions │ min_viewership_millions │ max_viewership_millions │
│    varchar    │    int64     │         double          │         double          │         double          │
├───────────────┼──────────────┼─────────────────────────┼─────────────────────────┼─────────────────────────┤
│ ABC | ESPN    │            5 │                    5.64 │                    3.61 │                    6.71 │
│ NBC/Peacock   │            1 │                    4.52 │                    4.52 │                    4.52 │
│ ABC           │           31 │                    2.31 │                    1.59 │                    3.34 │
│ NBC | Peacock │           50 │                    2.14 │                    1.05 │                    4.87 │
│ NBC           │            3 │                    2.05 │                    1.72 │                    2.47 │
│

In [9]:
con.sql("""
    SELECT
        national_tv,
        CASE
            WHEN national_tv IN ('NBC', 'NBC/Peacock', 'NBC | Peacock')
                THEN 'NBC | Peacock'
            ELSE national_tv
        END AS network_clean,
        COUNT(*) AS measurements
    FROM master_audience
    WHERE national_tv IS NOT NULL
    GROUP BY
        national_tv,
        network_clean
    ORDER BY
        network_clean,
        national_tv
""")

┌───────────────────────┬───────────────────────┬──────────────┐
│      national_tv      │     network_clean     │ measurements │
│        varchar        │        varchar        │    int64     │
├───────────────────────┼───────────────────────┼──────────────┤
│ ABC                   │ ABC                   │           31 │
│ ABC | ESPN            │ ABC | ESPN            │            5 │
│ ESPN                  │ ESPN                  │          110 │
│ NBC                   │ NBC | Peacock         │            3 │
│ NBC | Peacock         │ NBC | Peacock         │           85 │
│ NBC/Peacock           │ NBC | Peacock         │            3 │
│ NBC/Peacock/Telemundo │ NBC/Peacock/Telemundo │            2 │
│ Prime Video           │ Prime Video           │           27 │
└───────────────────────┴───────────────────────┴──────────────┘

In [10]:
con.sql("""
    CREATE OR REPLACE VIEW audience_clean AS
    SELECT
        *,
        CASE
            WHEN national_tv IN ('NBC', 'NBC/Peacock', 'NBC | Peacock')
                THEN 'NBC | Peacock'
            ELSE national_tv
        END AS network_clean
    FROM master_audience
""")

In [11]:
con.sql("""
    SELECT
        network_clean,
        COUNT(*) AS measurements,
        ROUND(AVG(viewership) / 1000000, 2) AS avg_viewership_millions
    FROM audience_clean
    WHERE measurement_type = 'average'
      AND network_clean IS NOT NULL
    GROUP BY network_clean
    ORDER BY avg_viewership_millions DESC
""")

┌───────────────┬──────────────┬─────────────────────────┐
│ network_clean │ measurements │ avg_viewership_millions │
│    varchar    │    int64     │         double          │
├───────────────┼──────────────┼─────────────────────────┤
│ ABC | ESPN    │            5 │                    5.64 │
│ ABC           │           31 │                    2.31 │
│ NBC | Peacock │           54 │                    2.18 │
│ ESPN          │          104 │                    1.37 │
│ Prime Video   │           27 │                    1.34 │
└───────────────┴──────────────┴─────────────────────────┘

In [15]:
#VIEWERSHIP FIGURES BY TEAM
result = con.sql("""
    WITH team_games AS (
        SELECT
            away_team AS team,
            viewership,
            measurement_type,
            measurement_scope,
            network_clean
        FROM audience_clean
        WHERE away_team IS NOT NULL

        UNION ALL

        SELECT
            home_team AS team,
            viewership,
            measurement_type,
            measurement_scope,
            network_clean
        FROM audience_clean
        WHERE home_team IS NOT NULL
    )

    SELECT
        team,
        COUNT(*) AS audience_measurements,
        ROUND(AVG(viewership) / 1000000, 2) AS avg_viewership_millions,
        ROUND(MEDIAN(viewership) / 1000000, 2) AS median_viewership_millions,
        ROUND(MAX(viewership) / 1000000, 2) AS max_viewership_millions
    FROM team_games
    WHERE measurement_type = 'average'
      AND measurement_scope = 'game'
    GROUP BY team
    HAVING COUNT(*) >= 3
    ORDER BY avg_viewership_millions DESC
""").df()

result

,team,audience_measurements,avg_viewership_millions,median_viewership_millions,max_viewership_millions
0,BOS,4,2.53,2.05,4.52
1,LAL,11,2.24,1.88,4.52
2,DAL,3,2.07,1.93,2.60
3,LAL (Lakers),31,2.06,1.56,5.35
4,NYK (Knicks),28,2.02,1.67,6.40
5,OKC (Thunder),26,2.00,1.62,6.71
6,DEN,5,1.98,2.23,2.74
7,NYK,8,1.93,1.79,3.34
8,GSW (Warriors),26,1.92,1.41,6.11
9,DAL (Mavericks),18,1.90,1.60,6.11


In [17]:
#Team Name Clean Up
con.sql("""
    CREATE OR REPLACE VIEW audience_clean AS
    SELECT
        *,
        CASE
            WHEN national_tv IN ('NBC', 'NBC/Peacock', 'NBC | Peacock')
                THEN 'NBC | Peacock'
            ELSE national_tv
        END AS network_clean,

        CASE
            WHEN away_team LIKE 'BOS%' THEN 'BOS'
            WHEN away_team LIKE 'BKN%' THEN 'BKN'
            WHEN away_team LIKE 'NYK%' THEN 'NYK'
            WHEN away_team LIKE 'PHI%' THEN 'PHI'
            WHEN away_team LIKE 'TOR%' THEN 'TOR'
            WHEN away_team LIKE 'CHI%' THEN 'CHI'
            WHEN away_team LIKE 'CLE%' THEN 'CLE'
            WHEN away_team LIKE 'DET%' THEN 'DET'
            WHEN away_team LIKE 'IND%' THEN 'IND'
            WHEN away_team LIKE 'MIL%' THEN 'MIL'
            WHEN away_team LIKE 'ATL%' THEN 'ATL'
            WHEN away_team LIKE 'CHA%' THEN 'CHA'
            WHEN away_team LIKE 'MIA%' THEN 'MIA'
            WHEN away_team LIKE 'ORL%' THEN 'ORL'
            WHEN away_team LIKE 'WAS%' THEN 'WAS'
            WHEN away_team LIKE 'DEN%' THEN 'DEN'
            WHEN away_team LIKE 'MIN%' THEN 'MIN'
            WHEN away_team LIKE 'OKC%' THEN 'OKC'
            WHEN away_team LIKE 'POR%' THEN 'POR'
            WHEN away_team LIKE 'UTA%' THEN 'UTA'
            WHEN away_team LIKE 'GSW%' THEN 'GSW'
            WHEN away_team LIKE 'LAC%' THEN 'LAC'
            WHEN away_team LIKE 'LAL%' THEN 'LAL'
            WHEN away_team LIKE 'PHX%' THEN 'PHX'
            WHEN away_team LIKE 'SAC%' THEN 'SAC'
            WHEN away_team LIKE 'DAL%' THEN 'DAL'
            WHEN away_team LIKE 'HOU%' THEN 'HOU'
            WHEN away_team LIKE 'MEM%' THEN 'MEM'
            WHEN away_team LIKE 'NOP%' THEN 'NOP'
            WHEN away_team LIKE 'SAS%' THEN 'SAS'
            ELSE away_team
        END AS away_team_clean,

        CASE
            WHEN home_team LIKE 'BOS%' THEN 'BOS'
            WHEN home_team LIKE 'BKN%' THEN 'BKN'
            WHEN home_team LIKE 'NYK%' THEN 'NYK'
            WHEN home_team LIKE 'PHI%' THEN 'PHI'
            WHEN home_team LIKE 'TOR%' THEN 'TOR'
            WHEN home_team LIKE 'CHI%' THEN 'CHI'
            WHEN home_team LIKE 'CLE%' THEN 'CLE'
            WHEN home_team LIKE 'DET%' THEN 'DET'
            WHEN home_team LIKE 'IND%' THEN 'IND'
            WHEN home_team LIKE 'MIL%' THEN 'MIL'
            WHEN home_team LIKE 'ATL%' THEN 'ATL'
            WHEN home_team LIKE 'CHA%' THEN 'CHA'
            WHEN home_team LIKE 'MIA%' THEN 'MIA'
            WHEN home_team LIKE 'ORL%' THEN 'ORL'
            WHEN home_team LIKE 'WAS%' THEN 'WAS'
            WHEN home_team LIKE 'DEN%' THEN 'DEN'
            WHEN home_team LIKE 'MIN%' THEN 'MIN'
            WHEN home_team LIKE 'OKC%' THEN 'OKC'
            WHEN home_team LIKE 'POR%' THEN 'POR'
            WHEN home_team LIKE 'UTA%' THEN 'UTA'
            WHEN home_team LIKE 'GSW%' THEN 'GSW'
            WHEN home_team LIKE 'LAC%' THEN 'LAC'
            WHEN home_team LIKE 'LAL%' THEN 'LAL'
            WHEN home_team LIKE 'PHX%' THEN 'PHX'
            WHEN home_team LIKE 'SAC%' THEN 'SAC'
            WHEN home_team LIKE 'DAL%' THEN 'DAL'
            WHEN home_team LIKE 'HOU%' THEN 'HOU'
            WHEN home_team LIKE 'MEM%' THEN 'MEM'
            WHEN home_team LIKE 'NOP%' THEN 'NOP'
            WHEN home_team LIKE 'SAS%' THEN 'SAS'
            ELSE home_team
        END AS home_team_clean

    FROM master_audience
""")

In [18]:
con.sql("""
    SELECT DISTINCT
        away_team,
        away_team_clean
    FROM audience_clean
    ORDER BY away_team_clean, away_team
""").df()

,away_team,away_team_clean
0,ATL,ATL
1,ATL (Hawks),ATL
2,BOS,BOS
3,BOS (Celtics),BOS
4,CHA (Hornets),CHA
5,CHI,CHI
6,CLE,CLE
7,CLE (Cavaliers),CLE
8,DAL,DAL
9,DAL (Mavericks),DAL


In [20]:
team_result = con.sql("""
    WITH team_games AS (
        SELECT
            away_team_clean AS team,
            viewership
        FROM audience_clean
        WHERE away_team_clean IS NOT NULL
          AND measurement_type = 'average'
          AND measurement_scope = 'game'

        UNION ALL

        SELECT
            home_team_clean AS team,
            viewership
        FROM audience_clean
        WHERE home_team_clean IS NOT NULL
          AND measurement_type = 'average'
          AND measurement_scope = 'game'
    )

    SELECT
        team,
        COUNT(*) AS audience_measurements,
        ROUND(AVG(viewership) / 1000000, 2) AS avg_viewership_millions,
        ROUND(MEDIAN(viewership) / 1000000, 2) AS median_viewership_millions,
        ROUND(MAX(viewership) / 1000000, 2) AS max_viewership_millions
    FROM team_games
    GROUP BY team
    HAVING COUNT(*) >= 3
    ORDER BY avg_viewership_millions DESC
""").df()

team_result

,team,audience_measurements,avg_viewership_millions,median_viewership_millions,max_viewership_millions
0,LAL,42,2.11,1.61,5.35
1,NYK,36,2.00,1.67,6.40
2,OKC,34,1.95,1.72,6.71
3,DAL,21,1.92,1.69,6.11
4,GSW,31,1.90,1.43,6.11
5,BOS,27,1.80,1.51,4.52
6,DEN,27,1.79,1.64,3.61
7,HOU,30,1.79,1.54,5.35
8,SAS,30,1.76,1.41,6.71
9,CLE,26,1.67,1.39,6.40


In [21]:
team_result = con.sql("""
    WITH team_games AS (
        SELECT
            away_team_clean AS team,
            viewership
        FROM audience_clean
        WHERE away_team_clean IS NOT NULL
          AND measurement_type = 'average'
          AND measurement_scope = 'game'

        UNION ALL

        SELECT
            home_team_clean AS team,
            viewership
        FROM audience_clean
        WHERE home_team_clean IS NOT NULL
          AND measurement_type = 'average'
          AND measurement_scope = 'game'
    )

    SELECT
        team,
        COUNT(*) AS audience_measurements,
        ROUND(AVG(viewership) / 1000000, 2) AS avg_viewership_millions,
        ROUND(MEDIAN(viewership) / 1000000, 2) AS median_viewership_millions,
        ROUND(MAX(viewership) / 1000000, 2) AS max_viewership_millions
    FROM team_games
    GROUP BY team
    ORDER BY avg_viewership_millions DESC
""").df()

team_result

,team,audience_measurements,avg_viewership_millions,median_viewership_millions,max_viewership_millions
0,LAL,42,2.11,1.61,5.35
1,NYK,36,2.00,1.67,6.40
2,IND,2,1.98,1.98,1.98
3,OKC,34,1.95,1.72,6.71
4,DAL,21,1.92,1.69,6.11
5,GSW,31,1.90,1.43,6.11
6,BOS,27,1.80,1.51,4.52
7,DEN,27,1.79,1.64,3.61
8,HOU,30,1.79,1.54,5.35
9,SAS,30,1.76,1.41,6.71


In [22]:
con.sql("""
    WITH all_teams AS (
        SELECT DISTINCT away_team_clean AS team
        FROM audience_clean
        WHERE away_team_clean IS NOT NULL

        UNION

        SELECT DISTINCT home_team_clean AS team
        FROM audience_clean
        WHERE home_team_clean IS NOT NULL
    ),

    average_game_teams AS (
        SELECT DISTINCT away_team_clean AS team
        FROM audience_clean
        WHERE measurement_type = 'average'
          AND measurement_scope = 'game'

        UNION

        SELECT DISTINCT home_team_clean AS team
        FROM audience_clean
        WHERE measurement_type = 'average'
          AND measurement_scope = 'game'
    )

    SELECT team
    FROM all_teams
    WHERE team NOT IN (SELECT team FROM average_game_teams)
    ORDER BY team
""").df()

,team
0,UTA


In [24]:
#Creating core game-level analytical table using average and game measurements
con.sql("""
    CREATE OR REPLACE TABLE game_audience AS
    SELECT
        a.match_key,
        CAST(a.date AS DATE) AS game_date,

        a.away_team_clean AS away_team,
        a.home_team_clean AS home_team,

        a.viewership AS audience_viewers,
        ROUND(a.viewership / 1000000, 3) AS audience_millions,

        a.network_clean AS network,

        s.season_type,
        s.tip_et,

        DAYOFWEEK(CAST(a.date AS DATE)) AS day_of_week_num,
        DAYNAME(CAST(a.date AS DATE)) AS day_of_week,

        EXTRACT(HOUR FROM CAST(s.tip_et AS TIMESTAMP)) AS tip_hour,

        a.measurement_source,
        a.source AS audience_source,
        a.source_url

    FROM audience_clean a

    LEFT JOIN master_schedule s
        ON a.match_key = s.match_key

    WHERE a.measurement_type = 'average'
      AND a.measurement_scope = 'game'
""")

game_audience = con.sql("""
    SELECT *
    FROM game_audience
    ORDER BY game_date, tip_et
""").df()

print("Rows:", len(game_audience))
print("Columns:", len(game_audience.columns))

game_audience.head(10)

Rows: 210
Columns: 15


,match_key,game_date,away_team,home_team,audience_viewers,audience_millions,network,season_type,tip_et,day_of_week_num,day_of_week,tip_hour,measurement_source,audience_source,source_url
0,2025-10-21_HOU_OKC,2025-10-21,HOU,OKC,4868000.0,4.868,NBC | Peacock,Regular Season,2025-10-21T18:30:00-05:00,2,Tuesday,18,Nielsen Big Data,TV Media Blog,https://tvmediablog.substack.com/p/2025-26-nba...
1,2025-10-21_GSW_LAL,2025-10-21,GSW,LAL,3990000.0,3.990,NBC | Peacock,Regular Season,2025-10-21T21:00:00-05:00,2,Tuesday,21,Nielsen Big Data,TV Media Blog,https://tvmediablog.substack.com/p/2025-26-nba...
2,2025-10-22_CLE_NYK,2025-10-22,CLE,NYK,2166000.0,2.166,ESPN,Regular Season,2025-10-22T18:00:00-05:00,3,Wednesday,18,Nielsen Big Data,TV Media Blog,https://tvmediablog.substack.com/p/2025-26-nba...
3,2025-10-22_DAL_SAS,2025-10-22,SAS,DAL,2488000.0,2.488,ESPN,Regular Season,2025-10-22T20:30:00-05:00,3,Wednesday,20,Nielsen Big Data,TV Media Blog,https://tvmediablog.substack.com/p/2025-26-nba...
4,2025-10-23_IND_OKC,2025-10-23,IND,OKC,1980000.0,1.980,ESPN,Regular Season,2025-10-23T18:30:00-05:00,4,Thursday,18,None,Sports Media Watch,https://www.sportsmediawatch.com/sports-rating...
5,2025-10-23_IND_OKC,2025-10-23,OKC,IND,1975000.0,1.975,ESPN,Regular Season,2025-10-23T18:30:00-05:00,4,Thursday,18,Nielsen Big Data,TV Media Blog,https://tvmediablog.substack.com/p/2025-26-nba...
6,2025-10-23_DEN_GSW,2025-10-23,DEN,GSW,2230000.0,2.230,ESPN,Regular Season,2025-10-23T21:00:00-05:00,4,Thursday,21,None,Sports Media Watch,https://www.sportsmediawatch.com/sports-rating...
7,2025-10-23_DEN_GSW,2025-10-23,DEN,GSW,2228000.0,2.228,ESPN,Regular Season,2025-10-23T21:00:00-05:00,4,Thursday,21,Nielsen Big Data,TV Media Blog,https://tvmediablog.substack.com/p/2025-26-nba...
8,2025-10-24_BOS_NYK,2025-10-24,BOS,NYK,1170000.0,1.170,Prime Video,Regular Season,2025-10-24T18:30:00-05:00,5,Friday,18,Nielsen Big Data,TV Media Blog,https://tvmediablog.substack.com/p/2025-26-nba...
9,2025-10-24_LAL_MIN,2025-10-24,MIN,LAL,1330000.0,1.330,Prime Video,Regular Season,2025-10-24T21:00:00-05:00,5,Friday,21,Nielsen Big Data,TV Media Blog,https://tvmediablog.substack.com/p/2025-26-nba...


In [25]:
#Determining Unique Games
duplicate_games = con.sql("""
    SELECT
        match_key,
        COUNT(*) AS observations,
        STRING_AGG(DISTINCT audience_source, ' | ') AS sources,
        ROUND(MIN(audience_millions), 3) AS min_audience_millions,
        ROUND(MAX(audience_millions), 3) AS max_audience_millions
    FROM game_audience
    GROUP BY match_key
    HAVING COUNT(*) > 1
    ORDER BY observations DESC, match_key
""").df()

print("Games with multiple audience observations:", len(duplicate_games))
print("Total game-level observations:", len(game_audience))

duplicate_games.head(20)

Games with multiple audience observations: 71
Total game-level observations: 210


,match_key,observations,sources,min_audience_millions,max_audience_millions
0,2025-11-05_LAL_SAS,3,Sports Media Watch | TV Media Blog,1.346,1.560
1,2025-11-05_MIN_NYK,3,Sports Media Watch | TV Media Blog,1.220,1.380
2,2026-01-16_HOU_MIN,3,TV Media Blog | Sports Media Watch,1.320,1.530
3,2026-01-28_CLE_LAL,3,TV Media Blog | Sports Media Watch,1.580,1.884
4,2026-01-30_DET_GSW,3,TV Media Blog | Sports Media Watch,1.280,1.390
5,2026-02-01_BOS_MIL,3,TV Media Blog | Sports Media Watch,1.346,1.510
6,2026-03-04_NYK_OKC,3,Sports Media Watch | TV Media Blog,1.508,1.583
7,2026-03-06_BOS_DAL,3,TV Media Blog | Sports Media Watch,1.506,1.692
8,2026-03-11_CLE_ORL,3,Sports Media Watch | TV Media Blog,0.903,1.151
9,2026-03-11_DEN_HOU,3,TV Media Blog | Sports Media Watch,0.913,1.120


In [26]:
#Analyzing Differences among duplicate games, for which multiple viewership figures were obtained from different sources
duplicate_detail = con.sql("""
    SELECT
        match_key,
        COUNT(*) AS observations,
        ROUND(MIN(audience_millions), 3) AS min_audience_millions,
        ROUND(MAX(audience_millions), 3) AS max_audience_millions,
        ROUND(
            (MAX(audience_millions) - MIN(audience_millions))
            / MIN(audience_millions) * 100,
            1
        ) AS pct_difference
    FROM game_audience
    GROUP BY match_key
    HAVING COUNT(*) > 1
    ORDER BY pct_difference DESC
""").df()

print("Duplicated games:", len(duplicate_detail))
print()
print("Difference between highest and lowest reported audience:")
print()
duplicate_detail.head(20)

Duplicated games: 71

Difference between highest and lowest reported audience:



,match_key,observations,min_audience_millions,max_audience_millions,pct_difference
0,2026-03-11_CLE_ORL,3,0.903,1.151,27.5
1,2026-01-14_CLE_PHI,2,0.944,1.196,26.7
2,2026-03-11_DEN_HOU,3,0.913,1.120,22.7
3,2026-03-25_ATL_DET,3,0.892,1.074,20.4
4,2026-03-14_LAC_SAC,2,0.822,0.987,20.1
5,2026-01-21_CHA_CLE,2,0.779,0.931,19.5
6,2026-01-28_CLE_LAL,3,1.580,1.884,19.2
7,2026-04-01_BOS_MIA,2,1.236,1.464,18.4
8,2026-02-11_NYK_PHI,2,0.903,1.069,18.4
9,2025-11-26_GSW_HOU,2,1.724,2.031,17.8


In [27]:
con.sql("""
    SELECT
        match_key,
        game_date,
        away_team,
        home_team,
        audience_millions,
        network,
        measurement_source,
        audience_source,
        source_url
    FROM game_audience
    WHERE match_key IN (
        '2026-03-11_CLE_ORL',
        '2026-01-14_CLE_PHI',
        '2026-03-11_DEN_HOU',
        '2025-11-05_LAL_SAS',
        '2025-10-23_IND_OKC'
    )
    ORDER BY match_key, audience_millions
""").df()

,match_key,game_date,away_team,home_team,audience_millions,network,measurement_source,audience_source,source_url
0,2025-10-23_IND_OKC,2025-10-23,OKC,IND,1.975,ESPN,Nielsen Big Data,TV Media Blog,https://tvmediablog.substack.com/p/2025-26-nba...
1,2025-10-23_IND_OKC,2025-10-23,IND,OKC,1.980,ESPN,None,Sports Media Watch,https://www.sportsmediawatch.com/sports-rating...
2,2025-11-05_LAL_SAS,2025-11-05,SAS,LAL,1.346,ESPN,Nielsen Panel,TV Media Blog,https://tvmediablog.substack.com/p/2025-26-nba...
3,2025-11-05_LAL_SAS,2025-11-05,SAS,LAL,1.559,ESPN,Nielsen Big Data,TV Media Blog,https://tvmediablog.substack.com/p/2025-26-nba...
4,2025-11-05_LAL_SAS,2025-11-05,SAS,LAL,1.560,ESPN,None,Sports Media Watch,https://www.sportsmediawatch.com/sports-rating...
5,2026-01-14_CLE_PHI,2026-01-14,CLE,PHI,0.944,ESPN,Nielsen Panel,TV Media Blog,https://tvmediablog.substack.com/p/2025-26-nba...
6,2026-01-14_CLE_PHI,2026-01-14,CLE,PHI,1.196,ESPN,Nielsen Big Data,TV Media Blog,https://tvmediablog.substack.com/p/2025-26-nba...
7,2026-03-11_CLE_ORL,2026-03-11,CLE,ORL,0.903,ESPN,Nielsen Panel,TV Media Blog,https://tvmediablog.substack.com/p/2025-26-nba...
8,2026-03-11_CLE_ORL,2026-03-11,CLE,ORL,1.150,ESPN,None,Sports Media Watch,https://www.sportsmediawatch.com/sports-rating...
9,2026-03-11_CLE_ORL,2026-03-11,CLE,ORL,1.151,ESPN,Nielsen Big Data,TV Media Blog,https://tvmediablog.substack.com/p/2025-26-nba...


In [28]:
# Creating separate analytical table with one row per game, using Nielsen Big Data average audience where available.
con.sql("""
    SELECT
        COALESCE(measurement_source, 'Unspecified') AS measurement_source,
        COUNT(*) AS observations,
        COUNT(DISTINCT match_key) AS unique_games,
        ROUND(AVG(audience_millions), 3) AS avg_audience_millions
    FROM game_audience
    GROUP BY 1
    ORDER BY observations DESC
""").df()

,measurement_source,observations,unique_games,avg_audience_millions
0,Nielsen Big Data,120,120,1.911
1,Nielsen Panel,49,49,1.209
2,Unspecified,41,41,1.830


In [29]:
#Creating new game table game_audience_clean with one row per game
con.sql("""
    CREATE OR REPLACE TABLE game_audience_clean AS

    WITH ranked AS (
        SELECT
            *,
            ROW_NUMBER() OVER (
                PARTITION BY match_key
                ORDER BY
                    CASE
                        WHEN measurement_source = 'Nielsen Big Data' THEN 1
                        WHEN measurement_source = 'Nielsen Panel' THEN 2
                        ELSE 3
                    END,
                    audience_source
            ) AS rn

        FROM game_audience
    )

    SELECT
        match_key,
        game_date,

        away_team,
        home_team,

        audience_viewers,
        audience_millions,

        network,
        season_type,
        tip_et,

        day_of_week_num,
        day_of_week,
        tip_hour,

        measurement_source,
        audience_source,
        source_url

    FROM ranked
    WHERE rn = 1
""")

game_audience_clean = con.sql("""
    SELECT *
    FROM game_audience_clean
    ORDER BY game_date, tip_et
""").df()

print("Rows:", len(game_audience_clean))
print("Unique games:", game_audience_clean["match_key"].nunique())
print()
print(game_audience_clean["measurement_source"].value_counts(dropna=False))

Rows: 127
Unique games: 127

Nielsen Big Data    120
Nielsen Panel         4
None                  3
Name: measurement_source, dtype: int64


In [30]:
con.sql("""
    SELECT
        COUNT(*) AS total_games,
        COUNT(*) FILTER (WHERE audience_millions IS NULL) AS missing_audience,
        COUNT(*) FILTER (WHERE network IS NULL) AS missing_network,
        COUNT(*) FILTER (WHERE away_team IS NULL) AS missing_away_team,
        COUNT(*) FILTER (WHERE home_team IS NULL) AS missing_home_team,
        COUNT(*) FILTER (WHERE season_type IS NULL) AS missing_season_type,
        COUNT(*) FILTER (WHERE tip_et IS NULL) AS missing_tip_time,
        COUNT(*) FILTER (WHERE day_of_week IS NULL) AS missing_day,
        COUNT(*) FILTER (WHERE measurement_source IS NULL) AS missing_measurement_source
    FROM game_audience_clean
""").df()

,total_games,missing_audience,missing_network,missing_away_team,missing_home_team,missing_season_type,missing_tip_time,missing_day,missing_measurement_source
0,127,0,12,0,0,0,7,0,3


In [31]:
con.sql("""
    SELECT
        match_key,
        audience_viewers,
        audience_millions,
        network,
        away_team,
        home_team
    FROM game_audience_clean
    LIMIT 10
""").df()

,match_key,audience_viewers,audience_millions,network,away_team,home_team
0,2025-10-21_GSW_LAL,3990000.0,3.990,NBC | Peacock,GSW,LAL
1,2025-11-28_MIL_NYK,2111000.0,2.111,Prime Video,MIL,NYK
2,2026-02-03_CHI_MIL,1050000.0,1.050,ESPN,CHI,MIL
3,2025-11-07_DEN_GSW,1060000.0,1.060,Prime Video,GSW,DEN
4,2025-11-14_GSW_SAS,1434000.0,1.434,Prime Video,GSW,SAS
5,2026-02-27_CLE_DET,1539000.0,1.539,ESPN,CLE,DET
6,2026-03-01_DEN_MIN,2741000.0,2.741,ABC,MIN,DEN
7,2026-03-08_HOU_SAS,2472000.0,2.472,NBC | Peacock,HOU,SAS
8,2026-03-22_BOS_MIN,1884000.0,1.884,NBC | Peacock,MIN,BOS
9,2026-03-25_ATL_DET,1074000.0,1.074,ESPN,ATL,DET


In [33]:
#Core Audience Distribution
con.sql("""
    SELECT
        COUNT(*) AS games,
        ROUND(AVG(audience_millions), 3) AS avg_audience_millions,
        ROUND(MEDIAN(audience_millions), 3) AS median_audience_millions,
        ROUND(MIN(audience_millions), 3) AS min_audience_millions,
        ROUND(MAX(audience_millions), 3) AS max_audience_millions,
        ROUND(QUANTILE_CONT(audience_millions, 0.25), 3) AS p25_audience_millions,
        ROUND(QUANTILE_CONT(audience_millions, 0.75), 3) AS p75_audience_millions
    FROM game_audience_clean
""").df()

,games,avg_audience_millions,median_audience_millions,min_audience_millions,max_audience_millions,p25_audience_millions,p75_audience_millions
0,127,1.865,1.529,0.516,6.711,1.23,2.139


In [34]:
#Audience by Network 
con.sql("""
    SELECT
        network,
        COUNT(*) AS games,
        ROUND(AVG(audience_millions), 3) AS avg_audience_millions,
        ROUND(MEDIAN(audience_millions), 3) AS median_audience_millions,
        ROUND(MIN(audience_millions), 3) AS min_audience_millions,
        ROUND(MAX(audience_millions), 3) AS max_audience_millions
    FROM game_audience_clean
    GROUP BY network
    ORDER BY avg_audience_millions DESC
""").df()

,network,games,avg_audience_millions,median_audience_millions,min_audience_millions,max_audience_millions
0,ABC | ESPN,5,5.635,6.107,3.610,6.711
1,NBC | Peacock,20,2.465,2.258,1.054,4.868
2,ABC,17,2.341,2.409,1.799,3.344
3,None,12,1.460,1.346,0.998,2.622
4,ESPN,49,1.433,1.385,0.516,2.488
5,Prime Video,24,1.327,1.185,0.528,3.072


In [38]:
#Inspecting Missing Network games
missing_networks = con.sql("""
    SELECT
        g.match_key,
        g.game_date,
        g.away_team,
        g.home_team,
        g.audience_millions,
        g.network,
        s.national_tv AS schedule_network
    FROM game_audience_clean AS g
    LEFT JOIN master_schedule AS s
        ON g.match_key = s.match_key
    WHERE g.network IS NULL
    ORDER BY g.game_date
""").df()

missing_networks

,match_key,game_date,away_team,home_team,audience_millions,network,schedule_network
0,2025-12-30_DET_LAL,2025-12-30,DET,LAL,1.724,None,None
1,2026-01-07_BOS_DEN,2026-01-07,DEN,BOS,1.306,None,None
2,2026-01-16_CLE_PHI,2026-01-16,CLE,PHI,1.253,None,None
3,2026-01-30_ORL_TOR,2026-01-30,TOR,ORL,0.998,None,None
4,2026-01-30_DET_GSW,2026-01-30,DET,GSW,1.386,None,None
5,2026-02-11_GSW_SAS,2026-02-11,SAS,GSW,1.258,None,None
6,2026-02-25_DET_OKC,2026-02-25,OKC,DET,1.490,None,None
7,2026-03-01_BOS_PHI,2026-03-01,PHI,BOS,2.622,None,None
8,2026-03-28_DET_MIN,2026-03-28,DET,MIN,1.593,None,None
9,2026-04-01_BOS_MIA,2026-04-01,BOS,MIA,1.464,None,None


In [39]:
con.sql("""
    SELECT
        match_key,
        game_date,
        away_team,
        home_team,
        audience_millions,
        audience_source,
        source_url
    FROM game_audience_clean
    WHERE network IS NULL
    ORDER BY game_date
""").df()

,match_key,game_date,away_team,home_team,audience_millions,audience_source,source_url
0,2025-12-30_DET_LAL,2025-12-30,DET,LAL,1.724,TV Media Blog,https://tvmediablog.substack.com/p/2025-26-nba...
1,2026-01-07_BOS_DEN,2026-01-07,DEN,BOS,1.306,TV Media Blog,https://tvmediablog.substack.com/p/2025-26-nba...
2,2026-01-16_CLE_PHI,2026-01-16,CLE,PHI,1.253,TV Media Blog,https://tvmediablog.substack.com/p/2025-26-nba...
3,2026-01-30_DET_GSW,2026-01-30,DET,GSW,1.386,TV Media Blog,https://tvmediablog.substack.com/p/2025-26-nba...
4,2026-01-30_ORL_TOR,2026-01-30,TOR,ORL,0.998,TV Media Blog,https://tvmediablog.substack.com/p/2025-26-nba...
5,2026-02-11_GSW_SAS,2026-02-11,SAS,GSW,1.258,TV Media Blog,https://tvmediablog.substack.com/p/2025-26-nba...
6,2026-02-25_DET_OKC,2026-02-25,OKC,DET,1.490,TV Media Blog,https://tvmediablog.substack.com/p/2025-26-nba...
7,2026-03-01_BOS_PHI,2026-03-01,PHI,BOS,2.622,TV Media Blog,https://tvmediablog.substack.com/p/2025-26-nba...
8,2026-03-28_DET_MIN,2026-03-28,DET,MIN,1.593,TV Media Blog,https://tvmediablog.substack.com/p/2025-26-nba...
9,2026-04-01_BOS_MIA,2026-04-01,BOS,MIA,1.464,TV Media Blog,https://tvmediablog.substack.com/p/2025-26-nba...


In [40]:
#checking dataset2 for missing network info
con.sql("""
    SELECT
        match_key,
        date,
        away_team,
        home_team,
        national_tv,
        viewership
    FROM dataset2
    WHERE match_key IN (
        SELECT match_key
        FROM game_audience_clean
        WHERE network IS NULL
    )
    ORDER BY date
""").df()

,match_key,date,away_team,home_team,national_tv,viewership
0,2025-12-30_DET_LAL,2025-12-30,DET (Pistons),LAL (Lakers),None,1724000.0
1,2025-12-30_DET_LAL,2025-12-30,DET (Pistons),LAL (Lakers),None,2000000.0
2,2026-01-07_BOS_DEN,2026-01-07,DEN (Nuggets),BOS (Celtics),None,1306000.0
3,2026-01-07_BOS_DEN,2026-01-07,DEN (Nuggets),BOS (Celtics),None,1158000.0
4,2026-01-16_CLE_PHI,2026-01-16,CLE (Cavaliers),PHI (76ers),None,1253000.0
5,2026-01-16_CLE_PHI,2026-01-16,CLE (Cavaliers),PHI (76ers),None,1159000.0
6,2026-01-30_ORL_TOR,2026-01-30,TOR (Raptors),ORL (Magic),None,998000.0
7,2026-01-30_DET_GSW,2026-01-30,DET (Pistons),GSW (Warriors),None,1386000.0
8,2026-01-30_ORL_TOR,2026-01-30,TOR (Raptors),ORL (Magic),None,852000.0
9,2026-01-30_DET_GSW,2026-01-30,DET (Pistons),GSW (Warriors),None,1280000.0


In [47]:
#Fixing missing networks
con.sql("""
    CREATE OR REPLACE TABLE game_audience_clean AS

    SELECT
        *,
        CASE
            WHEN match_key = '2025-12-30_DET_LAL' THEN 'NBC | Peacock'
            WHEN match_key = '2026-01-07_BOS_DEN' THEN 'ESPN'
            WHEN match_key = '2026-01-16_CLE_PHI' THEN 'ESPN'
            WHEN match_key = '2026-01-30_ORL_TOR' THEN 'ESPN'
            WHEN match_key = '2026-01-30_DET_GSW' THEN 'ESPN'
            WHEN match_key = '2026-02-11_GSW_SAS' THEN 'ESPN'
            WHEN match_key = '2026-02-25_DET_OKC' THEN 'ESPN'
            WHEN match_key = '2026-03-01_BOS_PHI' THEN 'NBC | Peacock'
            WHEN match_key = '2026-03-28_DET_MIN' THEN 'ABC'
            WHEN match_key = '2026-04-01_BOS_MIA' THEN 'ESPN'
            WHEN match_key = '2026-04-08_ATL_CLE' THEN 'ESPN'
            WHEN match_key = '2026-04-08_DAL_PHX' THEN 'ESPN'
            ELSE network
        END AS network_fixed
    FROM game_audience_clean
""")

In [48]:
con.sql("""
    SELECT
        network,
        network_fixed,
        COUNT(*) AS games
    FROM game_audience_clean
    GROUP BY 1, 2
    ORDER BY network_fixed, network
""").df()

,network,network_fixed,games
0,ABC,ABC,17
1,None,ABC,1
2,ABC | ESPN,ABC | ESPN,5
3,ESPN,ESPN,49
4,None,ESPN,9
5,NBC | Peacock,NBC | Peacock,20
6,None,NBC | Peacock,2
7,Prime Video,Prime Video,24


In [49]:
#making network_fixed the actual network value
con.sql("""
    CREATE OR REPLACE TABLE game_audience_clean AS
    SELECT
        match_key,
        game_date,
        away_team,
        home_team,
        audience_viewers,
        audience_millions,
        network_fixed AS network,
        season_type,
        tip_et,
        day_of_week_num,
        day_of_week,
        tip_hour,
        measurement_source,
        audience_source,
        source_url
    FROM game_audience_clean
""")

print(con.sql("""
    SELECT
        network,
        COUNT(*) AS games,
        ROUND(AVG(audience_millions), 3) AS avg_audience_millions
    FROM game_audience_clean
    GROUP BY network
    ORDER BY avg_audience_millions DESC
""").df())

         network  games  avg_audience_millions
0     ABC | ESPN      5                  5.635
1  NBC | Peacock     22                  2.438
2            ABC     18                  2.299
3           ESPN     58                  1.410
4    Prime Video     24                  1.327


In [50]:
#Evaluating Distribution Within Network
network_summary = con.sql("""
    SELECT
        network,
        COUNT(*) AS games,
        ROUND(AVG(audience_millions), 3) AS avg_audience_millions,
        ROUND(MEDIAN(audience_millions), 3) AS median_audience_millions,
        ROUND(MIN(audience_millions), 3) AS min_audience_millions,
        ROUND(MAX(audience_millions), 3) AS max_audience_millions
    FROM game_audience_clean
    GROUP BY network
    ORDER BY avg_audience_millions DESC
""").df()

network_summary

,network,games,avg_audience_millions,median_audience_millions,min_audience_millions,max_audience_millions
0,ABC | ESPN,5,5.635,6.107,3.610,6.711
1,NBC | Peacock,22,2.438,2.258,1.054,4.868
2,ABC,18,2.299,2.296,1.593,3.344
3,ESPN,58,1.410,1.377,0.516,2.488
4,Prime Video,24,1.327,1.185,0.528,3.072


In [51]:
#Team Level Audience Relationship
team_summary = con.sql("""
    WITH team_games AS (
        SELECT
            away_team AS team,
            audience_millions
        FROM game_audience_clean

        UNION ALL

        SELECT
            home_team AS team,
            audience_millions
        FROM game_audience_clean
    )

    SELECT
        team,
        COUNT(*) AS games,
        ROUND(AVG(audience_millions), 3) AS avg_audience_millions,
        ROUND(MEDIAN(audience_millions), 3) AS median_audience_millions,
        ROUND(MIN(audience_millions), 3) AS min_audience_millions,
        ROUND(MAX(audience_millions), 3) AS max_audience_millions
    FROM team_games
    GROUP BY team
    ORDER BY avg_audience_millions DESC
""").df()

team_summary

,team,games,avg_audience_millions,median_audience_millions,min_audience_millions,max_audience_millions
0,LAL,25,2.242,1.724,1.149,5.349
1,NYK,23,2.186,1.989,1.069,6.400
2,OKC,21,2.144,1.799,0.982,6.711
3,GSW,20,2.118,1.964,0.967,6.107
4,DAL,14,2.097,1.800,1.054,6.107
5,HOU,18,1.981,1.594,0.916,5.349
6,IND,1,1.975,1.975,1.975,1.975
7,CLE,14,1.902,1.534,0.931,6.400
8,DEN,17,1.893,1.752,0.960,3.610
9,SAS,20,1.888,1.495,0.916,6.711


In [52]:
#Team Sample Coverage
team_network = con.sql("""
    WITH team_games AS (
        SELECT away_team AS team, network, audience_millions
        FROM game_audience_clean

        UNION ALL

        SELECT home_team AS team, network, audience_millions
        FROM game_audience_clean
    )

    SELECT
        team,
        COUNT(*) AS total_games,
        COUNT(CASE WHEN network = 'ABC' THEN 1 END) AS abc_games,
        COUNT(CASE WHEN network = 'ABC | ESPN' THEN 1 END) AS abc_espn_games,
        COUNT(CASE WHEN network = 'NBC | Peacock' THEN 1 END) AS nbc_games,
        COUNT(CASE WHEN network = 'ESPN' THEN 1 END) AS espn_games,
        COUNT(CASE WHEN network = 'Prime Video' THEN 1 END) AS prime_games,
        ROUND(AVG(audience_millions), 3) AS avg_audience_millions
    FROM team_games
    GROUP BY team
    ORDER BY avg_audience_millions DESC
""").df()

team_network

,team,total_games,abc_games,abc_espn_games,nbc_games,espn_games,prime_games,avg_audience_millions
0,LAL,25,5,1,5,7,7,2.242
1,NYK,23,5,1,4,7,6,2.186
2,OKC,21,4,1,4,8,4,2.144
3,GSW,20,4,1,4,7,4,2.118
4,DAL,14,2,1,4,6,1,2.097
5,HOU,18,3,1,4,7,3,1.981
6,IND,1,0,0,0,1,0,1.975
7,CLE,14,2,1,2,9,0,1.902
8,DEN,17,3,1,3,7,3,1.893
9,SAS,20,1,1,2,10,6,1.888


In [53]:
#Introducing the timing component
day_summary = con.sql("""
    SELECT
        day_of_week,
        COUNT(*) AS games,
        ROUND(AVG(audience_millions), 3) AS avg_audience_millions,
        ROUND(MEDIAN(audience_millions), 3) AS median_audience_millions,
        ROUND(MIN(audience_millions), 3) AS min_audience_millions,
        ROUND(MAX(audience_millions), 3) AS max_audience_millions
    FROM game_audience_clean
    GROUP BY day_of_week_num, day_of_week
    ORDER BY day_of_week_num
""").df()

day_summary

,day_of_week,games,avg_audience_millions,median_audience_millions,min_audience_millions,max_audience_millions
0,Sunday,22,2.456,2.450,0.528,4.522
1,Monday,5,1.509,1.529,1.054,1.866
2,Tuesday,11,2.101,1.724,0.736,4.868
3,Wednesday,42,1.427,1.368,0.931,2.488
4,Thursday,13,3.053,1.975,0.967,6.711
5,Friday,19,1.366,1.330,0.916,2.111
6,Saturday,15,1.771,1.933,0.516,2.595


In [54]:
tip_time_summary = con.sql("""
    SELECT
        CASE
            WHEN tip_hour < 18 THEN 'Before 6 PM'
            WHEN tip_hour < 20 THEN '6–8 PM'
            WHEN tip_hour < 22 THEN '8–10 PM'
            ELSE '10 PM+'
        END AS tip_window,
        COUNT(*) AS games,
        ROUND(AVG(audience_millions), 3) AS avg_audience_millions,
        ROUND(MEDIAN(audience_millions), 3) AS median_audience_millions,
        ROUND(MIN(audience_millions), 3) AS min_audience_millions,
        ROUND(MAX(audience_millions), 3) AS max_audience_millions
    FROM game_audience_clean
    GROUP BY 1
    ORDER BY
        CASE
            WHEN tip_window = 'Before 6 PM' THEN 1
            WHEN tip_window = '6–8 PM' THEN 2
            WHEN tip_window = '8–10 PM' THEN 3
            ELSE 4
        END
""").df()

tip_time_summary

,tip_window,games,avg_audience_millions,median_audience_millions,min_audience_millions,max_audience_millions
0,Before 6 PM,20,2.642,2.075,0.528,6.711
1,6–8 PM,45,1.766,1.464,0.916,4.868
2,8–10 PM,39,1.783,1.529,0.822,5.349
3,10 PM+,23,1.521,1.386,0.516,3.610


In [55]:
#Creating Indicator Variables
con.sql("""
    CREATE OR REPLACE TABLE game_features AS
    SELECT
        match_key,
        game_date,
        away_team,
        home_team,
        network,
        season_type,
        tip_et,
        day_of_week,
        tip_hour,
        audience_millions,

        CASE
            WHEN day_of_week IN ('Saturday', 'Sunday') THEN 1
            ELSE 0
        END AS is_weekend,

        CASE
            WHEN tip_hour >= 18 AND tip_hour < 22 THEN 1
            ELSE 0
        END AS is_prime_time,

        CASE
            WHEN tip_hour < 18 THEN 1
            ELSE 0
        END AS is_early_game,

        CASE
            WHEN away_team IN ('LAL', 'NYK', 'OKC', 'GSW', 'DAL')
              OR home_team IN ('LAL', 'NYK', 'OKC', 'GSW', 'DAL')
            THEN 1
            ELSE 0
        END AS has_high_exposure_team

    FROM game_audience_clean
""")

print(con.sql("""
    SELECT *
    FROM game_features
    LIMIT 5
""").df())

            match_key  game_date away_team home_team        network  \
0  2025-10-21_GSW_LAL 2025-10-21       GSW       LAL  NBC | Peacock   
1  2025-11-28_MIL_NYK 2025-11-28       MIL       NYK    Prime Video   
2  2026-02-03_CHI_MIL 2026-02-03       CHI       MIL           ESPN   
3  2025-11-07_DEN_GSW 2025-11-07       GSW       DEN    Prime Video   
4  2025-11-14_GSW_SAS 2025-11-14       GSW       SAS    Prime Video   

      season_type                     tip_et day_of_week  tip_hour  \
0  Regular Season  2025-10-21T21:00:00-05:00     Tuesday        21   
1  Regular Season  2025-11-28T19:30:00-05:00      Friday        19   
2  Regular Season  2026-02-03T20:00:00-05:00     Tuesday        20   
3  Regular Season  2025-11-07T22:00:00-05:00      Friday        22   
4  Regular Season  2025-11-14T21:30:00-05:00      Friday        21   

   audience_millions  is_weekend  is_prime_time  is_early_game  \
0              3.990           0              1              0   
1              2.111

In [56]:
#Creating high-exposure team variables
con.sql("""
    CREATE OR REPLACE TABLE game_features AS
    SELECT
        *,
        
        CASE
            WHEN away_team = 'LAL' OR home_team = 'LAL' THEN 1
            ELSE 0
        END AS has_lal,

        CASE
            WHEN away_team = 'NYK' OR home_team = 'NYK' THEN 1
            ELSE 0
        END AS has_nyk,

        CASE
            WHEN away_team = 'OKC' OR home_team = 'OKC' THEN 1
            ELSE 0
        END AS has_okc,

        CASE
            WHEN away_team = 'GSW' OR home_team = 'GSW' THEN 1
            ELSE 0
        END AS has_gsw,

        CASE
            WHEN away_team = 'DAL' OR home_team = 'DAL' THEN 1
            ELSE 0
        END AS has_dal,

        (
            CASE WHEN away_team = 'LAL' OR home_team = 'LAL' THEN 1 ELSE 0 END +
            CASE WHEN away_team = 'NYK' OR home_team = 'NYK' THEN 1 ELSE 0 END +
            CASE WHEN away_team = 'OKC' OR home_team = 'OKC' THEN 1 ELSE 0 END +
            CASE WHEN away_team = 'GSW' OR home_team = 'GSW' THEN 1 ELSE 0 END +
            CASE WHEN away_team = 'DAL' OR home_team = 'DAL' THEN 1 ELSE 0 END
        ) AS high_exposure_team_count

    FROM game_features
""")

print(con.sql("""
    SELECT
        match_key,
        away_team,
        home_team,
        has_lal,
        has_nyk,
        has_okc,
        has_gsw,
        has_dal,
        high_exposure_team_count
    FROM game_features
    LIMIT 10
#""").df())

            match_key away_team home_team  has_lal  has_nyk  has_okc  has_gsw  \
0  2025-10-21_GSW_LAL       GSW       LAL        1        0        0        1   
1  2025-11-28_MIL_NYK       MIL       NYK        0        1        0        0   
2  2026-02-03_CHI_MIL       CHI       MIL        0        0        0        0   
3  2025-11-07_DEN_GSW       GSW       DEN        0        0        0        1   
4  2025-11-14_GSW_SAS       GSW       SAS        0        0        0        1   
5  2026-02-27_CLE_DET       CLE       DET        0        0        0        0   
6  2026-03-01_DEN_MIN       MIN       DEN        0        0        0        0   
7  2026-03-08_HOU_SAS       HOU       SAS        0        0        0        0   
8  2026-03-22_BOS_MIN       MIN       BOS        0        0        0        0   
9  2026-03-25_ATL_DET       ATL       DET        0        0        0        0   

   has_dal  high_exposure_team_count  
0        0                         2  
1        0                    

In [57]:
#compare audience by network and high-exposure-team presence

network_team_analysis = con.sql("""
    SELECT
        network,
        high_exposure_team_count,
        COUNT(*) AS games,
        ROUND(AVG(audience_millions), 3) AS avg_audience_millions,
        ROUND(MEDIAN(audience_millions), 3) AS median_audience_millions
    FROM game_features
    GROUP BY network, high_exposure_team_count
    ORDER BY network, high_exposure_team_count
""").df()

network_team_analysis

,network,high_exposure_team_count,games,avg_audience_millions,median_audience_millions
0,ABC,0,3,2.248,2.409
1,ABC,1,10,2.215,2.075
2,ABC,2,5,2.498,2.473
3,ABC | ESPN,0,1,3.610,3.610
4,ABC | ESPN,1,3,6.153,6.400
5,ABC | ESPN,2,1,6.107,6.107
6,ESPN,0,26,1.222,1.215
7,ESPN,1,29,1.570,1.387
8,ESPN,2,3,1.492,1.456
9,NBC | Peacock,0,7,1.894,1.884


In [58]:
#multivariate model prep
#linear regression with audience as the outcome and network, high-exposure teams, weekend status, and tip time as predictors.
#model data table
model_data = con.sql("""
    SELECT
        audience_millions,
        network,
        high_exposure_team_count,
        is_weekend,
        is_prime_time,
        is_early_game
    FROM game_features
""").df()

model_data.head()

,audience_millions,network,high_exposure_team_count,is_weekend,is_prime_time,is_early_game
0,3.990,NBC | Peacock,2,0,1,0
1,2.111,Prime Video,1,0,1,0
2,1.050,ESPN,0,0,1,0
3,1.060,Prime Video,1,0,0,0
4,1.434,Prime Video,1,0,1,0


In [59]:
#running the regression
#After accounting for network, does having one or two high-exposure teams, weekend scheduling, or prime-time scheduling correspond to higher observed national TV audiences?
#ESPN treated as reference network by default
import statsmodels.formula.api as smf

model = smf.ols(
    """
    audience_millions ~ C(network)
    + high_exposure_team_count
    + is_weekend
    + is_prime_time
    + is_early_game
    """,
    data=model_data
).fit()

print(model.summary())

                            OLS Regression Results                            
Dep. Variable:      audience_millions   R-squared:                       0.744
Model:                            OLS   Adj. R-squared:                  0.727
Method:                 Least Squares   F-statistic:                     42.88
Date:                Mon, 28 Sep 2026   Prob (F-statistic):           1.90e-31
Time:                        17:38:52   Log-Likelihood:                -103.10
No. Observations:                 127   AIC:                             224.2
Df Residuals:                     118   BIC:                             249.8
Df Model:                           8                                         
Covariance Type:            nonrobust                                         
                                  coef    std err          t      P>|t|      [0.025      0.975]
-----------------------------------------------------------------------------------------------
Intercept         

In [60]:
#comparison - predicted vs actual audience for 10 games with highest actual audience
model_data["predicted_audience"] = model.predict(model_data)

model_data["residual"] = (
    model_data["audience_millions"]
    - model_data["predicted_audience"]
)

model_data.sort_values(
    "audience_millions",
    ascending=False
)[[
    "audience_millions",
    "predicted_audience",
    "residual",
    "network",
    "high_exposure_team_count"
]].head(10)

,audience_millions,predicted_audience,residual,network,high_exposure_team_count
37,6.711,5.739987,0.971013,ABC | ESPN,1
26,6.400,5.739987,0.660013,ABC | ESPN,1
27,6.107,6.142404,-0.035404,ABC | ESPN,2
112,5.349,5.596069,-0.247069,ABC | ESPN,1
82,4.868,2.399471,2.468529,NBC | Peacock,1
32,4.522,2.551336,1.970664,NBC | Peacock,1
0,3.990,2.801887,1.188113,NBC | Peacock,2
86,3.610,4.958553,-1.348553,ABC | ESPN,0
80,3.601,2.953753,0.647247,NBC | Peacock,2
43,3.344,2.720834,0.623166,ABC,2


In [61]:
#Inspecting NBC due to large observed residuals
model_data["predicted_audience"] = model.predict(model_data)
model_data["residual"] = (
    model_data["audience_millions"]
    - model_data["predicted_audience"]
)

model_data[
    model_data["network"] == "NBC | Peacock"
].sort_values(
    "residual",
    ascending=False
)[[
    "audience_millions",
    "predicted_audience",
    "residual",
    "high_exposure_team_count"
]]

,audience_millions,predicted_audience,residual,high_exposure_team_count
82,4.868,2.399471,2.468529,1
32,4.522,2.551336,1.970664,1
0,3.990,2.801887,1.188113,2
80,3.601,2.953753,0.647247,2
64,2.622,2.148920,0.473080,0
7,2.472,2.148920,0.323080,0
125,3.076,2.953753,0.122247,2
51,3.016,2.953753,0.062247,2
113,1.961,1.997054,-0.036054,0
103,2.442,2.551336,-0.109336,1


In [62]:
#Inspecting NBC's highest-viewer games
con.sql("""
    SELECT
        match_key,
        game_date,
        away_team,
        home_team,
        network,
        audience_millions,
        high_exposure_team_count
    FROM game_features
    WHERE network = 'NBC | Peacock'
      AND audience_millions >= 3.0
    ORDER BY audience_millions DESC
""").df()

,match_key,game_date,away_team,home_team,network,audience_millions,high_exposure_team_count
0,2025-10-21_HOU_OKC,2025-10-21,HOU,OKC,NBC | Peacock,4.868,1
1,2026-02-22_BOS_LAL,2026-02-22,BOS,LAL,NBC | Peacock,4.522,1
2,2025-10-21_GSW_LAL,2025-10-21,GSW,LAL,NBC | Peacock,3.990,2
3,2026-02-01_LAL_NYK,2026-02-01,LAL,NYK,NBC | Peacock,3.601,2
4,2026-03-29_NYK_OKC,2026-03-29,NYK,OKC,NBC | Peacock,3.076,2
5,2026-04-05_DAL_LAL,2026-04-05,LAL,DAL,NBC | Peacock,3.016,2


In [63]:
#controlling for opening night variable
con.sql("""
    CREATE OR REPLACE TABLE game_features AS
    SELECT
        *,
        CASE
            WHEN game_date = '2025-10-21'
            THEN 1
            ELSE 0
        END AS is_opening_night
    FROM game_features
""")

In [64]:
con.sql("""
    SELECT
        match_key,
        away_team,
        home_team,
        audience_millions,
        is_opening_night
    FROM game_features
    WHERE is_opening_night = 1
""").df()

,match_key,away_team,home_team,audience_millions,is_opening_night
0,2025-10-21_GSW_LAL,GSW,LAL,3.990,1
1,2025-10-21_HOU_OKC,HOU,OKC,4.868,1


In [65]:
#Re-running the model
model_data = con.sql("""
    SELECT
        audience_millions,
        network,
        high_exposure_team_count,
        is_weekend,
        is_prime_time,
        is_early_game,
        is_opening_night
    FROM game_features
""").df()

model_opening = smf.ols(
    """
    audience_millions ~ C(network)
    + high_exposure_team_count
    + is_weekend
    + is_prime_time
    + is_early_game
    + is_opening_night
    """,
    data=model_data
).fit()

print(model_opening.summary())

                            OLS Regression Results                            
Dep. Variable:      audience_millions   R-squared:                       0.797
Model:                            OLS   Adj. R-squared:                  0.781
Method:                 Least Squares   F-statistic:                     51.04
Date:                Mon, 28 Sep 2026   Prob (F-statistic):           2.12e-36
Time:                        18:01:13   Log-Likelihood:                -88.387
No. Observations:                 127   AIC:                             196.8
Df Residuals:                     117   BIC:                             225.2
Df Model:                           9                                         
Covariance Type:            nonrobust                                         
                                  coef    std err          t      P>|t|      [0.025      0.975]
-----------------------------------------------------------------------------------------------
Intercept         

In [66]:
#Looking at average audience by team again to see if high exposure teams should be adjusted
team_audience = con.sql("""
    WITH team_games AS (
        SELECT away_team AS team, audience_millions
        FROM game_features

        UNION ALL

        SELECT home_team AS team, audience_millions
        FROM game_features
    )

    SELECT
        team,
        COUNT(*) AS games,
        ROUND(AVG(audience_millions), 3) AS avg_audience_millions,
        ROUND(MEDIAN(audience_millions), 3) AS median_audience_millions
    FROM team_games
    GROUP BY team
    ORDER BY avg_audience_millions DESC
""").df()

team_audience

,team,games,avg_audience_millions,median_audience_millions
0,LAL,25,2.242,1.724
1,NYK,23,2.186,1.989
2,OKC,21,2.144,1.799
3,GSW,20,2.118,1.964
4,DAL,14,2.097,1.800
5,HOU,18,1.981,1.594
6,IND,1,1.975,1.975
7,CLE,14,1.902,1.534
8,DEN,17,1.893,1.752
9,SAS,20,1.888,1.495


In [67]:
#testing for individual team effects outside of the high exposure group, using Boston as example
model_boston_data = con.sql("""
    SELECT
        audience_millions,
        network,
        high_exposure_team_count,
        is_weekend,
        is_prime_time,
        is_early_game,
        is_opening_night,
        CASE
            WHEN away_team = 'BOS' OR home_team = 'BOS'
            THEN 1
            ELSE 0
        END AS has_boston
    FROM game_features
""").df()

model_boston = smf.ols(
    """
    audience_millions ~ C(network)
    + high_exposure_team_count
    + has_boston
    + is_weekend
    + is_prime_time
    + is_early_game
    + is_opening_night
    """,
    data=model_boston_data
).fit()

print(model_boston.summary())

                            OLS Regression Results                            
Dep. Variable:      audience_millions   R-squared:                       0.804
Model:                            OLS   Adj. R-squared:                  0.788
Method:                 Least Squares   F-statistic:                     47.72
Date:                Mon, 28 Sep 2026   Prob (F-statistic):           1.85e-36
Time:                        18:09:35   Log-Likelihood:                -86.010
No. Observations:                 127   AIC:                             194.0
Df Residuals:                     116   BIC:                             225.3
Df Model:                          10                                         
Covariance Type:            nonrobust                                         
                                  coef    std err          t      P>|t|      [0.025      0.975]
-----------------------------------------------------------------------------------------------
Intercept         

In [68]:
#comparing boston model to initial model
print("Previous model:")
print(f"R²: {model_opening.rsquared:.3f}")
print(f"Adjusted R²: {model_opening.rsquared_adj:.3f}")
print(f"AIC: {model_opening.aic:.1f}")

print("\nBoston model:")
print(f"R²: {model_boston.rsquared:.3f}")
print(f"Adjusted R²: {model_boston.rsquared_adj:.3f}")
print(f"AIC: {model_boston.aic:.1f}")

print("\nBoston coefficient:")
print(f"{model_boston.params['has_boston']:.3f} million viewers")
print(f"p-value: {model_boston.pvalues['has_boston']:.3f}")

Previous model:
R²: 0.797
Adjusted R²: 0.781
AIC: 196.8

Boston model:
R²: 0.804
Adjusted R²: 0.788
AIC: 194.0

Boston coefficient:
0.299 million viewers
p-value: 0.038


In [70]:
#5-fold cross validation
import statsmodels.api as sm
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error, r2_score
import numpy as np

# Use the same data as the Boston model
cv_data = model_boston_data.copy()

# Create dummy variables for categorical network
cv_data = pd.get_dummies(
    cv_data,
    columns=["network"],
    drop_first=True,
    dtype=float
)

X = cv_data.drop(columns=["audience_millions"])
y = cv_data["audience_millions"]

# 5-fold cross-validation
kf = KFold(n_splits=5, shuffle=True, random_state=42)

rmse_scores = []
r2_scores = []

for train_idx, test_idx in kf.split(X):
    X_train = X.iloc[train_idx]
    X_test = X.iloc[test_idx]
    y_train = y.iloc[train_idx]
    y_test = y.iloc[test_idx]

    X_train = sm.add_constant(X_train)
    X_test = sm.add_constant(X_test, has_constant="add")

    cv_model = sm.OLS(y_train, X_train).fit()

    predictions = cv_model.predict(X_test)

    rmse_scores.append(
        np.sqrt(mean_squared_error(y_test, predictions))
    )
    r2_scores.append(
        r2_score(y_test, predictions)
    )

print("5-Fold Cross-Validation")
print("------------------------")
print(f"Average RMSE: {np.mean(rmse_scores):.3f}M")
print(f"RMSE by fold: {[round(x, 3) for x in rmse_scores]}")
print(f"Average R²: {np.mean(r2_scores):.3f}")
print(f"R² by fold: {[round(x, 3) for x in r2_scores]}")

5-Fold Cross-Validation
------------------------
Average RMSE: 0.540M
RMSE by fold: [0.5, 0.575, 0.39, 0.576, 0.661]
Average R²: 0.712
R² by fold: [0.828, 0.675, 0.816, 0.482, 0.762]


In [71]:
#boston effect: whether it survives heteroskedasticity-robust standard errors
model_boston_robust = model_boston.get_robustcov_results(
    cov_type="HC3"
)

print(model_boston_robust.summary())

                            OLS Regression Results                            
Dep. Variable:      audience_millions   R-squared:                       0.804
Model:                            OLS   Adj. R-squared:                  0.788
Method:                 Least Squares   F-statistic:                     24.17
Date:                Mon, 28 Sep 2026   Prob (F-statistic):           5.17e-24
Time:                        18:49:45   Log-Likelihood:                -86.010
No. Observations:                 127   AIC:                             194.0
Df Residuals:                     116   BIC:                             225.3
Df Model:                          10                                         
Covariance Type:                  HC3                                         
                                  coef    std err          t      P>|t|      [0.025      0.975]
-----------------------------------------------------------------------------------------------
Intercept         

In [72]:
#log audience model - proportional differences in audience rather than absolute difference
import numpy as np
import statsmodels.formula.api as smf

log_model_data = model_boston_data.copy()

log_model_data["log_audience"] = np.log(
    log_model_data["audience_millions"]
)

model_log = smf.ols(
    """
    log_audience ~ C(network)
    + high_exposure_team_count
    + has_boston
    + is_weekend
    + is_prime_time
    + is_early_game
    + is_opening_night
    """,
    data=log_model_data
).fit(
    cov_type="HC3"
)

print(model_log.summary())

                            OLS Regression Results                            
Dep. Variable:           log_audience   R-squared:                       0.699
Model:                            OLS   Adj. R-squared:                  0.673
Method:                 Least Squares   F-statistic:                     37.36
Date:                Mon, 28 Sep 2026   Prob (F-statistic):           1.04e-31
Time:                        18:52:49   Log-Likelihood:                -5.2285
No. Observations:                 127   AIC:                             32.46
Df Residuals:                     116   BIC:                             63.74
Df Model:                          10                                         
Covariance Type:                  HC3                                         
                                  coef    std err          z      P>|z|      [0.025      0.975]
-----------------------------------------------------------------------------------------------
Intercept         

In [73]:
#5-fold cross validation
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error, r2_score
import numpy as np

cv_log_data = log_model_data.copy()

cv_log_data = pd.get_dummies(
    cv_log_data,
    columns=["network"],
    drop_first=True,
    dtype=float
)

X = cv_log_data.drop(
    columns=["audience_millions", "log_audience"]
)

y = cv_log_data["log_audience"]

kf = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

rmse_log_scores = []
r2_log_scores = []

for train_idx, test_idx in kf.split(X):

    X_train = X.iloc[train_idx]
    X_test = X.iloc[test_idx]

    y_train = y.iloc[train_idx]
    y_test = y.iloc[test_idx]

    X_train = sm.add_constant(X_train)
    X_test = sm.add_constant(X_test, has_constant="add")

    cv_model = sm.OLS(y_train, X_train).fit()

    predictions_log = cv_model.predict(X_test)

    rmse_log_scores.append(
        np.sqrt(mean_squared_error(y_test, predictions_log))
    )

    r2_log_scores.append(
        r2_score(y_test, predictions_log)
    )

print("5-Fold Cross-Validation — Log Audience")
print("---------------------------------------")
print(f"Average RMSE: {np.mean(rmse_log_scores):.3f}")
print(f"RMSE by fold: {[round(x, 3) for x in rmse_log_scores]}")
print(f"Average R²: {np.mean(r2_log_scores):.3f}")
print(f"R² by fold: {[round(x, 3) for x in r2_log_scores]}")

5-Fold Cross-Validation — Log Audience
---------------------------------------
Average RMSE: 0.286
RMSE by fold: [0.221, 0.308, 0.237, 0.284, 0.381]
Average R²: 0.581
R² by fold: [0.802, 0.423, 0.629, 0.472, 0.576]


In [74]:
#testing model predictions in actual viewer units - 15 games where model is the most wrong
prediction_check = model_boston_data.copy()

prediction_check["predicted_audience"] = model_boston.predict(
    model_boston_data
)

prediction_check["residual"] = (
    prediction_check["audience_millions"]
    - prediction_check["predicted_audience"]
)

prediction_check["absolute_error"] = (
    prediction_check["residual"].abs()
)

prediction_check = prediction_check.sort_values(
    "absolute_error",
    ascending=False
)

prediction_check[
    [
        "audience_millions",
        "predicted_audience",
        "residual",
        "absolute_error",
        "network",
        "high_exposure_team_count",
        "has_boston",
        "is_opening_night"
    ]
].head(15)

,audience_millions,predicted_audience,residual,absolute_error,network,high_exposure_team_count,has_boston,is_opening_night
25,3.072,1.193253,1.878747,1.878747,Prime Video,1,0,0
32,4.522,2.671578,1.850422,1.850422,NBC | Peacock,1,1,0
86,3.610,5.032997,-1.422997,1.422997,ABC | ESPN,0,0,0
37,6.711,5.712246,0.998754,0.998754,ABC | ESPN,1,0,0
76,2.488,1.522329,0.965671,0.965671,ESPN,1,0,0
84,2.433,1.522329,0.910671,0.910671,ESPN,1,0,0
6,2.741,1.864476,0.876524,0.876524,ABC,0,0,0
117,1.054,1.899461,-0.845461,0.845461,NBC | Peacock,1,0,0
106,0.528,1.371790,-0.843790,0.843790,Prime Video,0,0,0
80,3.601,2.778613,0.822387,0.822387,NBC | Peacock,2,0,0


In [77]:
#identifying the actual games
# Add game identifiers to the prediction table
prediction_check_named = con.sql("""
    SELECT
        match_key,
        game_date,
        away_team,
        home_team,
        audience_millions,
        network,
        high_exposure_team_count,

        CASE
            WHEN away_team = 'BOS' OR home_team = 'BOS'
            THEN 1
            ELSE 0
        END AS has_boston,

        is_opening_night

    FROM game_features
""").df()

# Add model predictions
prediction_check_named["predicted_audience"] = model_boston.predict(
    model_boston_data
)

# Calculate residuals
prediction_check_named["residual"] = (
    prediction_check_named["audience_millions"]
    - prediction_check_named["predicted_audience"]
)

prediction_check_named["absolute_error"] = (
    prediction_check_named["residual"].abs()
)

# Largest prediction errors
prediction_check_named = prediction_check_named.sort_values(
    "absolute_error",
    ascending=False
)

prediction_check_named[
    [
        "match_key",
        "game_date",
        "away_team",
        "home_team",
        "audience_millions",
        "predicted_audience",
        "residual",
        "absolute_error",
        "network",
        "high_exposure_team_count",
        "has_boston",
        "is_opening_night"
    ]
].head(15)

,match_key,game_date,away_team,home_team,audience_millions,predicted_audience,residual,absolute_error,network,high_exposure_team_count,has_boston,is_opening_night
25,2025-12-16_NYK_SAS,2025-12-16,SAS,NYK,3.072,1.193253,1.878747,1.878747,Prime Video,1,0,0
32,2026-02-22_BOS_LAL,2026-02-22,BOS,LAL,4.522,2.671578,1.850422,1.850422,NBC | Peacock,1,1,0
86,2025-12-25_DEN_MIN,2025-12-25,MIN,DEN,3.610,5.032997,-1.422997,1.422997,ABC | ESPN,0,0,0
37,2025-12-25_OKC_SAS,2025-12-25,SAS,OKC,6.711,5.712246,0.998754,0.998754,ABC | ESPN,1,0,0
76,2025-10-22_DAL_SAS,2025-10-22,SAS,DAL,2.488,1.522329,0.965671,0.965671,ESPN,1,0,0
84,2025-11-26_MIN_OKC,2025-11-26,MIN,OKC,2.433,1.522329,0.910671,0.910671,ESPN,1,0,0
6,2026-03-01_DEN_MIN,2026-03-01,MIN,DEN,2.741,1.864476,0.876524,0.876524,ABC,0,0,0
117,2025-12-29_DAL_POR,2025-12-29,DAL,POR,1.054,1.899461,-0.845461,0.845461,NBC | Peacock,1,0,0
106,2026-01-18_MEM_ORL,2026-01-18,ORL,MEM,0.528,1.371790,-0.843790,0.843790,Prime Video,0,0,0
80,2026-02-01_LAL_NYK,2026-02-01,LAL,NYK,3.601,2.778613,0.822387,0.822387,NBC | Peacock,2,0,0


In [78]:
#Adding Christmas and NBA Cup Identifiers

con.sql("""
    CREATE OR REPLACE TABLE game_features AS
    SELECT
        *,
        
        CASE
            WHEN game_date = '2025-12-25'
            THEN 1
            ELSE 0
        END AS is_christmas,

        CASE
            WHEN season_type = 'NBA Cup'
            THEN 1
            ELSE 0
        END AS is_nba_cup

    FROM game_features
""")

# Verify the new variables
con.sql("""
    SELECT
        is_christmas,
        is_nba_cup,
        COUNT(*) AS games,
        ROUND(AVG(audience_millions), 3) AS avg_audience_millions,
        ROUND(MEDIAN(audience_millions), 3) AS median_audience_millions
    FROM game_features
    GROUP BY is_christmas, is_nba_cup
    ORDER BY is_christmas DESC, is_nba_cup DESC
""").df()

,is_christmas,is_nba_cup,games,avg_audience_millions,median_audience_millions
0,1,0,5,5.635,6.107
1,0,1,5,1.317,1.330
2,0,0,117,1.727,1.529


In [80]:
#inspecting nba cup playoff games
con.sql("""
    SELECT
        match_key,
        game_date,
        away_team,
        home_team,
        tip_et,
        audience_millions,
        network,
        high_exposure_team_count,
        is_opening_night,
        is_christmas,
        is_nba_cup
    FROM game_features
    WHERE is_nba_cup = 1
    ORDER BY game_date
""").df()

,match_key,game_date,away_team,home_team,tip_et,audience_millions,network,high_exposure_team_count,is_opening_night,is_christmas,is_nba_cup
0,2025-12-09_MIA_ORL,2025-12-09,MIA,ORL,None,0.736,Prime Video,0,0,0,1
1,2025-12-10_LAL_SAS,2025-12-10,SAS,LAL,None,1.460,Prime Video,1,0,0,1
2,2025-12-10_OKC_PHX,2025-12-10,PHX,OKC,None,1.050,Prime Video,1,0,0,1
3,2025-12-13_OKC_SAS,2025-12-13,SAS,OKC,None,2.010,Prime Video,1,0,0,1
4,2025-12-13_NYK_ORL,2025-12-13,NYK,ORL,None,1.330,Prime Video,1,0,0,1


In [82]:
#adding two games to nba cup variable 
con.sql("""
    UPDATE game_features
    SET is_nba_cup = 1
    WHERE match_key IN (
        '2025-12-09_NYK_TOR',
        '2025-12-16_NYK_SAS'
    )
""")

con.sql("""
    SELECT
        match_key,
        game_date,
        away_team,
        home_team,
        tip_et,
        audience_millions,
        network,
        is_nba_cup
    FROM game_features
    WHERE is_nba_cup = 1
    ORDER BY game_date
""").df()  

,match_key,game_date,away_team,home_team,tip_et,audience_millions,network,is_nba_cup
0,2025-12-09_NYK_TOR,2025-12-09,NYK,TOR,None,1.140,Prime Video,1
1,2025-12-09_MIA_ORL,2025-12-09,MIA,ORL,None,0.736,Prime Video,1
2,2025-12-10_LAL_SAS,2025-12-10,SAS,LAL,None,1.460,Prime Video,1
3,2025-12-10_OKC_PHX,2025-12-10,PHX,OKC,None,1.050,Prime Video,1
4,2025-12-13_OKC_SAS,2025-12-13,SAS,OKC,None,2.010,Prime Video,1
5,2025-12-13_NYK_ORL,2025-12-13,NYK,ORL,None,1.330,Prime Video,1
6,2025-12-16_NYK_SAS,2025-12-16,SAS,NYK,None,3.072,Prime Video,1


In [83]:
#Inspecting Christmas games by tip time: 
con.sql("""
    SELECT
        match_key,
        game_date,
        away_team,
        home_team,
        tip_et,
        tip_hour,
        audience_millions,
        network,
        is_christmas
    FROM game_features
    WHERE is_christmas = 1
    ORDER BY tip_et
""").df()

,match_key,game_date,away_team,home_team,tip_et,tip_hour,audience_millions,network,is_christmas
0,2025-12-25_CLE_NYK,2025-12-25,CLE,NYK,2025-12-25T12:00:00-05:00,12,6.400,ABC | ESPN,1
1,2025-12-25_OKC_SAS,2025-12-25,SAS,OKC,2025-12-25T14:30:00-05:00,14,6.711,ABC | ESPN,1
2,2025-12-25_DAL_GSW,2025-12-25,DAL,GSW,2025-12-25T17:00:00-05:00,17,6.107,ABC | ESPN,1
3,2025-12-25_HOU_LAL,2025-12-25,HOU,LAL,2025-12-25T20:00:00-05:00,20,5.349,ABC | ESPN,1
4,2025-12-25_DEN_MIN,2025-12-25,MIN,DEN,2025-12-25T22:30:00-05:00,22,3.610,ABC | ESPN,1


In [84]:
#Specifying the nba cup final 
con.sql("""
    ALTER TABLE game_features
    ADD COLUMN IF NOT EXISTS is_nba_cup_final INTEGER
""")

con.sql("""
    UPDATE game_features
    SET is_nba_cup_final = CASE
        WHEN match_key = '2025-12-16_NYK_SAS' THEN 1
        ELSE 0
    END
""")

con.sql("""
    SELECT
        match_key,
        game_date,
        away_team,
        home_team,
        audience_millions,
        network,
        is_nba_cup,
        is_nba_cup_final
    FROM game_features
    WHERE is_nba_cup = 1
    ORDER BY game_date
""").df()

,match_key,game_date,away_team,home_team,audience_millions,network,is_nba_cup,is_nba_cup_final
0,2025-12-09_NYK_TOR,2025-12-09,NYK,TOR,1.140,Prime Video,1,0
1,2025-12-09_MIA_ORL,2025-12-09,MIA,ORL,0.736,Prime Video,1,0
2,2025-12-10_LAL_SAS,2025-12-10,SAS,LAL,1.460,Prime Video,1,0
3,2025-12-10_OKC_PHX,2025-12-10,PHX,OKC,1.050,Prime Video,1,0
4,2025-12-13_OKC_SAS,2025-12-13,SAS,OKC,2.010,Prime Video,1,0
5,2025-12-13_NYK_ORL,2025-12-13,NYK,ORL,1.330,Prime Video,1,0
6,2025-12-16_NYK_SAS,2025-12-16,SAS,NYK,3.072,Prime Video,1,1


In [86]:
#creating variable for other nba cup games
con.sql("""
    ALTER TABLE game_features
    ADD COLUMN IF NOT EXISTS is_nba_cup_other INTEGER
""")

con.sql("""
    UPDATE game_features
    SET is_nba_cup_other = CASE
        WHEN is_nba_cup = 1 AND is_nba_cup_final = 0 THEN 1
        ELSE 0
    END
""")

con.sql("""
    SELECT
        is_nba_cup_other,
        is_nba_cup_final,
        COUNT(*) AS games,
        ROUND(AVG(audience_millions), 3) AS avg_audience_millions,
        ROUND(MEDIAN(audience_millions), 3) AS median_audience_millions
    FROM game_features
    GROUP BY is_nba_cup_other, is_nba_cup_final
    ORDER BY is_nba_cup_other DESC, is_nba_cup_final DESC
""").df()

,is_nba_cup_other,is_nba_cup_final,games,avg_audience_millions,median_audience_millions
0,1,0,6,1.288,1.235
1,0,1,1,3.072,3.072
2,0,0,120,1.884,1.539


In [88]:
#Model with variables for NBA Cup Playoff games and Christmas
model_data_events = con.sql("""
    SELECT
        audience_millions,
        network,
        high_exposure_team_count,
        CASE
            WHEN away_team = 'BOS' OR home_team = 'BOS' THEN 1
            ELSE 0
        END AS has_boston,
        is_weekend,
        is_prime_time,
        is_early_game,
        is_opening_night,
        is_christmas,
        is_nba_cup_other,
        is_nba_cup_final
    FROM game_features
""").df()

model_events = smf.ols(
    """audience_millions ~ C(network)
    + high_exposure_team_count
    + has_boston
    + is_weekend
    + is_prime_time
    + is_early_game
    + is_opening_night
    + is_christmas
    + is_nba_cup_other
    + is_nba_cup_final""",
    data=model_data_events
).fit(cov_type="HC3")

print(model_events.summary())

#takeaway - all 5 ABC | ESPN games are the Christmas games; Christmas identifier not needed independently for Network analysis

                            OLS Regression Results                            
Dep. Variable:      audience_millions   R-squared:                       0.832
Model:                            OLS   Adj. R-squared:                  0.814
Method:                 Least Squares   F-statistic:                     23.54
Date:                Mon, 28 Sep 2026   Prob (F-statistic):           1.72e-25
Time:                        19:46:30   Log-Likelihood:                -76.430
No. Observations:                 127   AIC:                             178.9
Df Residuals:                     114   BIC:                             215.8
Df Model:                          12                                         
Covariance Type:                  HC3                                         
                                  coef    std err          z      P>|z|      [0.025      0.975]
-----------------------------------------------------------------------------------------------
Intercept         

/Users/drewbeal/anaconda3/lib/python3.10/site-packages/statsmodels/base/model.py:1871: ValueWarning: covariance of constraints does not have full rank. The number of constraints is 13, but rank is 12
  warnings.warn('covariance of constraints does not have full '


In [89]:
#model comparing NBA Cup vs Christmas
christmas_model_data = con.sql("""
    SELECT
        audience_millions,
        high_exposure_team_count,
        CASE
            WHEN away_team = 'BOS' OR home_team = 'BOS' THEN 1
            ELSE 0
        END AS has_boston,
        is_weekend,
        is_prime_time,
        is_early_game,
        is_opening_night,
        is_christmas,
        is_nba_cup_other,
        is_nba_cup_final
    FROM game_features
""").df()

christmas_model = smf.ols(
    """audience_millions ~ high_exposure_team_count
    + has_boston
    + is_weekend
    + is_prime_time
    + is_early_game
    + is_opening_night
    + is_christmas
    + is_nba_cup_other
    + is_nba_cup_final""",
    data=christmas_model_data
).fit(cov_type="HC3")

print(christmas_model.summary())

                            OLS Regression Results                            
Dep. Variable:      audience_millions   R-squared:                       0.794
Model:                            OLS   Adj. R-squared:                  0.778
Method:                 Least Squares   F-statistic:                     26.33
Date:                Mon, 28 Sep 2026   Prob (F-statistic):           2.83e-24
Time:                        19:51:03   Log-Likelihood:                -89.254
No. Observations:                 127   AIC:                             198.5
Df Residuals:                     117   BIC:                             226.9
Df Model:                           9                                         
Covariance Type:                  HC3                                         
                               coef    std err          z      P>|z|      [0.025      0.975]
--------------------------------------------------------------------------------------------
Intercept               

In [90]:
#Simple event-level comparison
con.sql("""
    SELECT
        CASE
            WHEN is_christmas = 1 THEN 'Christmas'
            WHEN is_nba_cup_final = 1 THEN 'NBA Cup Final'
            WHEN is_nba_cup_other = 1 THEN 'NBA Cup - Other'
            ELSE 'Regular Season'
        END AS event_type,
        COUNT(*) AS games,
        ROUND(AVG(audience_millions), 3) AS avg_audience_millions,
        ROUND(MEDIAN(audience_millions), 3) AS median_audience_millions,
        ROUND(MIN(audience_millions), 3) AS min_audience_millions,
        ROUND(MAX(audience_millions), 3) AS max_audience_millions
    FROM game_features
    GROUP BY 1
    ORDER BY
        CASE
            WHEN event_type = 'Regular Season' THEN 1
            WHEN event_type = 'NBA Cup - Other' THEN 2
            WHEN event_type = 'NBA Cup Final' THEN 3
            WHEN event_type = 'Christmas' THEN 4
        END
""").df()

,event_type,games,avg_audience_millions,median_audience_millions,min_audience_millions,max_audience_millions
0,Regular Season,115,1.720,1.529,0.516,4.868
1,NBA Cup - Other,6,1.288,1.235,0.736,2.010
2,NBA Cup Final,1,3.072,3.072,3.072,3.072
3,Christmas,5,5.635,6.107,3.610,6.711


In [ ]:
#Examining Star Player Availability and Impact on Audience

In [94]:
#Initial Test Case - Cooper Flagg and the Dallas Mavericks
con.sql("""
    SELECT
        match_key,
        game_date,
        away_team,
        home_team,
        tip_et,
        audience_millions,
        network,
        high_exposure_team_count,
        is_weekend,
        is_prime_time,
        is_early_game,
        is_opening_night,
        is_christmas,
        is_nba_cup_other,
        is_nba_cup_final
    FROM game_features
    WHERE away_team = 'DAL'
       OR home_team = 'DAL'
    ORDER BY game_date
""").df()

,match_key,game_date,away_team,home_team,tip_et,audience_millions,network,high_exposure_team_count,is_weekend,is_prime_time,is_early_game,is_opening_night,is_christmas,is_nba_cup_other,is_nba_cup_final
0,2025-10-22_DAL_SAS,2025-10-22,SAS,DAL,2025-10-22T20:30:00-05:00,2.488,ESPN,1,0,1,0,0,0,0,0
1,2025-11-19_DAL_NYK,2025-11-19,NYK,DAL,2025-11-19T21:30:00-05:00,1.436,ESPN,2,0,1,0,0,0,0,0
2,2025-11-28_DAL_LAL,2025-11-28,DAL,LAL,2025-11-28T22:00:00-05:00,2.065,Prime Video,2,0,0,0,0,0,0,0
3,2025-12-23_DAL_DEN,2025-12-23,DEN,DAL,2025-12-23T20:00:00-05:00,1.734,NBC | Peacock,1,0,1,0,0,0,0,0
4,2025-12-25_DAL_GSW,2025-12-25,DAL,GSW,2025-12-25T17:00:00-05:00,6.107,ABC | ESPN,2,0,0,1,0,1,0,0
5,2025-12-29_DAL_POR,2025-12-29,DAL,POR,2025-12-29T23:00:00-05:00,1.054,NBC | Peacock,1,0,0,0,0,0,0,0
6,2026-01-14_DAL_DEN,2026-01-14,DEN,DAL,2026-01-14T21:30:00-05:00,1.059,ESPN,1,0,1,0,0,0,0,0
7,2026-01-19_DAL_NYK,2026-01-19,DAL,NYK,2026-01-19T17:00:00-05:00,1.866,NBC | Peacock,2,0,0,1,0,0,0,0
8,2026-01-24_DAL_LAL,2026-01-24,LAL,DAL,2026-01-24T20:30:00-05:00,2.595,ABC,2,1,1,0,0,0,0,0
9,2026-01-31_DAL_HOU,2026-01-31,DAL,HOU,2026-01-31T20:30:00-05:00,1.933,ABC,1,1,1,0,0,0,0,0


In [95]:
#creating flagg_absent variable
con.sql("""
    CREATE OR REPLACE TABLE game_features AS
    SELECT
        *,
        CASE
            WHEN game_date IN ('2025-11-19', '2026-02-20') THEN 1
            ELSE 0
        END AS flagg_absent
    FROM game_features
""")

In [96]:
con.sql("""
    SELECT
        match_key,
        game_date,
        away_team,
        home_team,
        audience_millions,
        flagg_absent
    FROM game_features
    WHERE away_team = 'DAL'
       OR home_team = 'DAL'
    ORDER BY game_date
""").df()

,match_key,game_date,away_team,home_team,audience_millions,flagg_absent
0,2025-10-22_DAL_SAS,2025-10-22,SAS,DAL,2.488,0
1,2025-11-19_DAL_NYK,2025-11-19,NYK,DAL,1.436,1
2,2025-11-28_DAL_LAL,2025-11-28,DAL,LAL,2.065,0
3,2025-12-23_DAL_DEN,2025-12-23,DEN,DAL,1.734,0
4,2025-12-25_DAL_GSW,2025-12-25,DAL,GSW,6.107,0
5,2025-12-29_DAL_POR,2025-12-29,DAL,POR,1.054,0
6,2026-01-14_DAL_DEN,2026-01-14,DEN,DAL,1.059,0
7,2026-01-19_DAL_NYK,2026-01-19,DAL,NYK,1.866,0
8,2026-01-24_DAL_LAL,2026-01-24,LAL,DAL,2.595,0
9,2026-01-31_DAL_HOU,2026-01-31,DAL,HOU,1.933,0


In [98]:
#flagg model prep - testing only the 14 nationally televised Mavs games; using smaller number of variables
dallas_model_data = con.sql("""
    SELECT
        audience_millions,
        flagg_absent,
        is_weekend,
        is_prime_time,
        is_christmas
    FROM game_features
    WHERE away_team = 'DAL'
       OR home_team = 'DAL'
""").df()

print(dallas_model_data)
print("\nNumber of Dallas games:", len(dallas_model_data))
print("Flagg absent:", dallas_model_data["flagg_absent"].sum())

    audience_millions  flagg_absent  is_weekend  is_prime_time  is_christmas
0               6.107             0           0              0             1
1               1.933             0           1              1             0
2               3.016             0           1              1             0
3               1.229             0           0              1             0
4               1.866             0           0              0             0
5               1.692             0           0              1             0
6               2.065             0           0              0             0
7               1.059             0           0              1             0
8               2.488             0           0              1             0
9               1.734             0           0              1             0
10              1.087             1           0              1             0
11              1.436             1           0              1             0

In [99]:
dallas_summary = (
    dallas_model_data
    .groupby("flagg_absent")["audience_millions"]
    .agg(["count", "mean", "median", "min", "max"])
    .reset_index()
)

dallas_summary["flagg_status"] = dallas_summary["flagg_absent"].map({
    0: "Flagg played",
    1: "Flagg absent"
})

dallas_summary = dallas_summary[
    ["flagg_status", "count", "mean", "median", "min", "max"]
]

print(dallas_summary)

   flagg_status  count    mean  median    min    max
0  Flagg played     12  2.2365  1.8995  1.054  6.107
1  Flagg absent      2  1.2615  1.2615  1.087  1.436


In [100]:
dallas_comparison = con.sql("""
    SELECT
        match_key,
        game_date,
        away_team,
        home_team,
        audience_millions,
        network,
        flagg_absent,
        is_weekend,
        is_prime_time,
        is_christmas
    FROM game_features
    WHERE away_team = 'DAL'
       OR home_team = 'DAL'
    ORDER BY game_date
""").df()

dallas_comparison

,match_key,game_date,away_team,home_team,audience_millions,network,flagg_absent,is_weekend,is_prime_time,is_christmas
0,2025-10-22_DAL_SAS,2025-10-22,SAS,DAL,2.488,ESPN,0,0,1,0
1,2025-11-19_DAL_NYK,2025-11-19,NYK,DAL,1.436,ESPN,1,0,1,0
2,2025-11-28_DAL_LAL,2025-11-28,DAL,LAL,2.065,Prime Video,0,0,0,0
3,2025-12-23_DAL_DEN,2025-12-23,DEN,DAL,1.734,NBC | Peacock,0,0,1,0
4,2025-12-25_DAL_GSW,2025-12-25,DAL,GSW,6.107,ABC | ESPN,0,0,0,1
5,2025-12-29_DAL_POR,2025-12-29,DAL,POR,1.054,NBC | Peacock,0,0,0,0
6,2026-01-14_DAL_DEN,2026-01-14,DEN,DAL,1.059,ESPN,0,0,1,0
7,2026-01-19_DAL_NYK,2026-01-19,DAL,NYK,1.866,NBC | Peacock,0,0,0,0
8,2026-01-24_DAL_LAL,2026-01-24,LAL,DAL,2.595,ABC,0,1,1,0
9,2026-01-31_DAL_HOU,2026-01-31,DAL,HOU,1.933,ABC,0,1,1,0


In [102]:
dallas_predictions = con.sql("""
    SELECT
        match_key,
        game_date,
        away_team,
        home_team,
        audience_millions,
        network,
        flagg_absent,
        high_exposure_team_count,
        CASE
            WHEN away_team = 'BOS' OR home_team = 'BOS' THEN 1
            ELSE 0
        END AS has_boston,
        is_weekend,
        is_prime_time,
        is_early_game,
        is_opening_night,
        is_christmas,
        is_nba_cup_other,
        is_nba_cup_final
    FROM game_features
    WHERE away_team = 'DAL'
       OR home_team = 'DAL'
""").df()

dallas_predictions["predicted_audience"] = base_model.predict(
    dallas_predictions
)

dallas_predictions["residual"] = (
    dallas_predictions["audience_millions"]
    - dallas_predictions["predicted_audience"]
)

dallas_predictions[
    [
        "match_key",
        "audience_millions",
        "predicted_audience",
        "residual",
        "flagg_absent"
    ]
].sort_values("game_date")

KeyError: 'game_date'

In [103]:
#Mavericks Audience Prediction Residuals w/ Flagg Absence
dallas_predictions[
    [
        "match_key",
        "game_date",
        "audience_millions",
        "predicted_audience",
        "residual",
        "flagg_absent"
    ]
].sort_values("game_date")

,match_key,game_date,audience_millions,predicted_audience,residual,flagg_absent
8,2025-10-22_DAL_SAS,2025-10-22,2.488,1.537009,0.950991,0
11,2025-11-19_DAL_NYK,2025-11-19,1.436,1.946168,-0.510168,1
6,2025-11-28_DAL_LAL,2025-11-28,2.065,1.385433,0.679567,0
9,2025-12-23_DAL_DEN,2025-12-23,1.734,2.081934,-0.347934,0
0,2025-12-25_DAL_GSW,2025-12-25,6.107,6.147091,-0.040091,0
12,2025-12-29_DAL_POR,2025-12-29,1.054,1.798407,-0.744407,0
7,2026-01-14_DAL_DEN,2026-01-14,1.059,1.537009,-0.478009,0
4,2026-01-19_DAL_NYK,2026-01-19,1.866,2.605658,-0.739658,0
13,2026-01-24_DAL_LAL,2026-01-24,2.595,2.564826,0.030174,0
1,2026-01-31_DAL_HOU,2026-01-31,1.933,2.155667,-0.222667,0


In [104]:
#back to python to build dataframe for availability of other star players
star_players = {
    "BOS": ["Jaylen Brown", "Jayson Tatum"],
    "CLE": ["Donovan Mitchell"],
    "DET": ["Cade Cunningham"],
    "MIL": ["Giannis Antetokounmpo"],
    "NYK": ["Jalen Brunson"],
    "ORL": ["Paolo Banchero"],
    "PHI": ["Joel Embiid", "Tyrese Maxey"],
    "DAL": ["Cooper Flagg"],
    "DEN": ["Nikola Jokic"],
    "GSW": ["Stephen Curry"],
    "HOU": ["Kevin Durant"],
    "LAC": ["Kawhi Leonard"],
    "LAL": ["Luka Doncic", "LeBron James"],
    "MIN": ["Anthony Edwards"],
    "PHX": ["Devin Booker"],
    "SAS": ["Victor Wembanyama"],
    "OKC": ["Shai Gilgeous-Alexander"],
    "MEM": ["Ja Morant"]
}

star_player_count = sum(len(players) for players in star_players.values())

print("Teams:", len(star_players))
print("Players:", star_player_count)

for team, players in star_players.items():
    print(team, ":", ", ".join(players))

Teams: 18
Players: 21
BOS : Jaylen Brown, Jayson Tatum
CLE : Donovan Mitchell
DET : Cade Cunningham
MIL : Giannis Antetokounmpo
NYK : Jalen Brunson
ORL : Paolo Banchero
PHI : Joel Embiid, Tyrese Maxey
DAL : Cooper Flagg
DEN : Nikola Jokic
GSW : Stephen Curry
HOU : Kevin Durant
LAC : Kawhi Leonard
LAL : Luka Doncic, LeBron James
MIN : Anthony Edwards
PHX : Devin Booker
SAS : Victor Wembanyama
OKC : Shai Gilgeous-Alexander
MEM : Ja Morant


In [105]:
#testing player availability scrape automation with Kevin Durant
import pandas as pd

url = "https://www.basketball-reference.com/players/d/duranke01/gamelog-advanced/2026/"

kd_log = pd.read_html(url)[0]

print(kd_log.shape)
display(kd_log.head())

(89, 24)


,Rk,Gcar,Gtm,Date,Team,Unnamed: 5,Opp,Result,GS,MP,...,TRB%,AST%,STL%,BLK%,TOV%,USG%,ORtg,DRtg,GmSc,BPM
0,1,1124,1,2025-10-21,HOU,@,OKC,"L, 124-125 (2OT)",*,47:03,...,12.3,11.6,0.0,0.0,17.7,20.6,112,114,13.4,-2.1
1,2,1125,2,2025-10-24,HOU,NaN,DET,"L, 111-115",*,36:48,...,4.2,6.4,4.0,2.6,0.0,28.6,161,116,31.7,18.5
2,3,1126,3,2025-10-27,HOU,NaN,BRK,"W, 137-109",*,24:48,...,15.3,15.5,2.1,0.0,14.9,36.5,109,113,11.9,-11.6
3,4,1127,4,2025-10-29,HOU,@,TOR,"W, 139-121",*,36:45,...,8.7,3.7,0.0,8.7,8.2,26.9,139,123,22.4,2.4
4,Rk,Gcar,Gtm,Date,Team,NaN,Opp,Result,GS,MP,...,TRB%,AST%,STL%,BLK%,TOV%,USG%,ORtg,DRtg,GmSc,BPM


In [107]:
#cleaning game log
kd_log_clean = kd_log.copy()

# Remove repeated header rows
kd_log_clean = kd_log_clean[kd_log_clean["Date"] != "Date"].copy()

# Convert date
kd_log_clean["Date"] = pd.to_datetime(kd_log_clean["Date"])

# Inspect unusual MP values
valid_mp = kd_log_clean["MP"].astype(str).str.match(r"^\d+:\d+$")
print("Unusual MP values:")
print(kd_log_clean.loc[~valid_mp, "MP"].value_counts())

# Convert minutes played safely
def minutes_to_decimal(x):
    if pd.isna(x):
        return 0
    x = str(x)
    if ":" not in x:
        return 0
    minutes, seconds = x.split(":")
    return int(minutes) + int(seconds) / 60

kd_log_clean["minutes"] = kd_log_clean["MP"].apply(minutes_to_decimal)

# Played = positive minutes
kd_log_clean["played"] = (kd_log_clean["minutes"] > 0).astype(int)

print("\nRows:", len(kd_log_clean))
print("Games played:", kd_log_clean["played"].sum())
print("Games absent:", (kd_log_clean["played"] == 0).sum())

display(
    kd_log_clean[
        ["Date", "Team", "Opp", "MP", "minutes", "played"]
    ].head(10)
)

Unusual MP values:
Inactive    4
2841        1
Name: MP, dtype: int64

Rows: 83
Games played: 78
Games absent: 5


,Date,Team,Opp,MP,minutes,played
0,2025-10-21,HOU,OKC,47:03,47.050000,1
1,2025-10-24,HOU,DET,36:48,36.800000,1
2,2025-10-27,HOU,BRK,24:48,24.800000,1
3,2025-10-29,HOU,TOR,36:45,36.750000,1
5,2025-11-01,HOU,BOS,29:33,29.550000,1
6,2025-11-03,HOU,DAL,38:24,38.400000,1
7,2025-11-05,HOU,MEM,34:27,34.450000,1
8,2025-11-07,HOU,SAS,36:33,36.550000,1
9,2025-11-09,HOU,MIL,37:21,37.350000,1
10,2025-11-12,HOU,WAS,30:52,30.866667,1


In [108]:
#KD Inactive games
kd_absences = kd_log_clean[
    kd_log_clean["MP"] == "Inactive"
][["Date", "Team", "Opp"]].copy()

print("Kevin Durant inactive games:", len(kd_absences))
display(kd_absences)

Kevin Durant inactive games: 4


,Date,Team,Opp
15,2025-11-24,HOU,PHO
16,2025-11-26,HOU,GSW
51,2026-02-02,HOU,IND
87,2026-04-12,HOU,MEM


In [109]:
#confirming basketball reference player ID's
br_player_ids = {
    "Jaylen Brown": "brownja02",
    "Jayson Tatum": "tatumja01",
    "Donovan Mitchell": "mitchdo01",
    "Cade Cunningham": "cunnica01",
    "Giannis Antetokounmpo": "antetgi01",
    "Jalen Brunson": "brunsja01",
    "Paolo Banchero": "banchpa01",
    "Joel Embiid": "embiijo01",
    "Tyrese Maxey": "maxeyty01",
    "Cooper Flagg": "flaggco01",
    "Nikola Jokic": "jokicni01",
    "Stephen Curry": "curryst01",
    "Kevin Durant": "duranke01",
    "Kawhi Leonard": "leonaka01",
    "Luka Doncic": "doncilu01",
    "LeBron James": "jamesle01",
    "Anthony Edwards": "edwaran01",
    "Devin Booker": "bookede01",
    "Victor Wembanyama": "wembavi01",
    "Shai Gilgeous-Alexander": "gilgesh01",
    "Ja Morant": "moranja01"
}

print("Players:", len(br_player_ids))
for player, player_id in br_player_ids.items():
    print(f"{player}: {player_id}")

Players: 21
Jaylen Brown: brownja02
Jayson Tatum: tatumja01
Donovan Mitchell: mitchdo01
Cade Cunningham: cunnica01
Giannis Antetokounmpo: antetgi01
Jalen Brunson: brunsja01
Paolo Banchero: banchpa01
Joel Embiid: embiijo01
Tyrese Maxey: maxeyty01
Cooper Flagg: flaggco01
Nikola Jokic: jokicni01
Stephen Curry: curryst01
Kevin Durant: duranke01
Kawhi Leonard: leonaka01
Luka Doncic: doncilu01
LeBron James: jamesle01
Anthony Edwards: edwaran01
Devin Booker: bookede01
Victor Wembanyama: wembavi01
Shai Gilgeous-Alexander: gilgesh01
Ja Morant: moranja01


In [110]:
#Building 127 target game set for search, to line up with available national tv dataset
target_games = con.sql("""
    SELECT
        match_key,
        game_date,
        away_team,
        home_team
    FROM game_features
    ORDER BY game_date
""").df()

print("Target games:", len(target_games))
display(target_games.head(10))

Target games: 127


,match_key,game_date,away_team,home_team
0,2025-10-21_HOU_OKC,2025-10-21,HOU,OKC
1,2025-10-21_GSW_LAL,2025-10-21,GSW,LAL
2,2025-10-22_CLE_NYK,2025-10-22,CLE,NYK
3,2025-10-22_DAL_SAS,2025-10-22,SAS,DAL
4,2025-10-23_IND_OKC,2025-10-23,OKC,IND
5,2025-10-23_DEN_GSW,2025-10-23,DEN,GSW
6,2025-10-24_BOS_NYK,2025-10-24,BOS,NYK
7,2025-10-24_LAL_MIN,2025-10-24,MIN,LAL
8,2025-10-29_LAL_MIN,2025-10-29,LAL,MIN
9,2025-10-29_BOS_CLE,2025-10-29,CLE,BOS


In [111]:
#Building star player absence dataset
import time

availability_records = []

for player, player_id in br_player_ids.items():
    # Find the player's team from our star_players dictionary
    player_team = next(
        team for team, players in star_players.items()
        if player in players
    )

    url = f"https://www.basketball-reference.com/players/{player_id[0]}/{player_id}/gamelog-advanced/2026/"
    
    try:
        player_log = pd.read_html(url)[0]

        # Remove repeated header rows
        player_log = player_log[player_log["Date"] != "Date"].copy()
        player_log["Date"] = pd.to_datetime(player_log["Date"], errors="coerce")

        # Keep only valid dates and the player's designated team
        player_log = player_log[
            player_log["Date"].notna() &
            (player_log["Team"] == player_team)
        ].copy()

        # Keep only games in our national-TV dataset
        merged = target_games.merge(
            player_log[["Date", "MP"]],
            left_on="game_date",
            right_on="Date",
            how="left"
        )

        # Only evaluate games involving the player's team
        merged = merged[
            (merged["away_team"] == player_team) |
            (merged["home_team"] == player_team)
        ].copy()

        for _, row in merged.iterrows():
            mp = row["MP"]

            if pd.isna(mp):
                played = None
            else:
                played = 0 if mp == "Inactive" else 1

            availability_records.append({
                "match_key": row["match_key"],
                "game_date": row["game_date"],
                "team": player_team,
                "player": player,
                "mp": mp,
                "played": played,
                "absent": None if played is None else 1 - played,
                "source": "Basketball-Reference",
                "source_url": url
            })

        print(f"{player}: {len(merged)} target games processed")

    except Exception as e:
        print(f"ERROR - {player}: {e}")

    time.sleep(1)

star_availability = pd.DataFrame(availability_records)

print("\nTotal records:", len(star_availability))
print("Players:", star_availability["player"].nunique())
print("Games represented:", star_availability["match_key"].nunique())

display(star_availability.head(15))

Jaylen Brown: 16 target games processed
Jayson Tatum: 16 target games processed
Donovan Mitchell: 14 target games processed
Cade Cunningham: 8 target games processed
Giannis Antetokounmpo: 5 target games processed
Jalen Brunson: 23 target games processed
Paolo Banchero: 6 target games processed
Joel Embiid: 7 target games processed
Tyrese Maxey: 7 target games processed
Cooper Flagg: 14 target games processed
Nikola Jokic: 17 target games processed
Stephen Curry: 20 target games processed
Kevin Durant: 18 target games processed
Kawhi Leonard: 7 target games processed
Luka Doncic: 25 target games processed
LeBron James: 25 target games processed
Anthony Edwards: 13 target games processed
Devin Booker: 3 target games processed
Victor Wembanyama: 20 target games processed
Shai Gilgeous-Alexander: 21 target games processed
Ja Morant: 3 target games processed

Total records: 288
Players: 21
Games represented: 127


,match_key,game_date,team,player,mp,played,absent,source,source_url
0,2025-10-24_BOS_NYK,2025-10-24,BOS,Jaylen Brown,35:28,1.0,0.0,Basketball-Reference,https://www.basketball-reference.com/players/b...
1,2025-10-29_BOS_CLE,2025-10-29,BOS,Jaylen Brown,31:50,1.0,0.0,Basketball-Reference,https://www.basketball-reference.com/players/b...
2,2025-10-31_BOS_PHI,2025-10-31,BOS,Jaylen Brown,31:44,1.0,0.0,Basketball-Reference,https://www.basketball-reference.com/players/b...
3,2025-11-26_BOS_DET,2025-11-26,BOS,Jaylen Brown,34:13,1.0,0.0,Basketball-Reference,https://www.basketball-reference.com/players/b...
4,2026-01-07_BOS_DEN,2026-01-07,BOS,Jaylen Brown,34:15,1.0,0.0,Basketball-Reference,https://www.basketball-reference.com/players/b...
5,2026-01-19_BOS_DET,2026-01-19,BOS,Jaylen Brown,38:55,1.0,0.0,Basketball-Reference,https://www.basketball-reference.com/players/b...
6,2026-02-01_BOS_MIL,2026-02-01,BOS,Jaylen Brown,29:29,1.0,0.0,Basketball-Reference,https://www.basketball-reference.com/players/b...
7,2026-02-08_BOS_NYK,2026-02-08,BOS,Jaylen Brown,33:08,1.0,0.0,Basketball-Reference,https://www.basketball-reference.com/players/b...
8,2026-02-22_BOS_LAL,2026-02-22,BOS,Jaylen Brown,36:01,1.0,0.0,Basketball-Reference,https://www.basketball-reference.com/players/b...
9,2026-02-25_BOS_DEN,2026-02-25,BOS,Jaylen Brown,33:46,1.0,0.0,Basketball-Reference,https://www.basketball-reference.com/players/b...


In [112]:
#check for missing statuses
# Check for games where we could not determine whether the player played
missing_status = star_availability[
    star_availability["played"].isna()
].copy()

print("Records with unknown status:", len(missing_status))

if len(missing_status) > 0:
    display(
        missing_status[
            ["match_key", "game_date", "team", "player", "mp"]
        ]
    )

print("\nAbsences by player:")
absence_summary = (
    star_availability
    .groupby("player")
    .agg(
        target_games=("match_key", "count"),
        absences=("absent", "sum")
    )
    .sort_values("absences", ascending=False)
)

display(absence_summary)

Records with unknown status: 5


,match_key,game_date,team,player,mp
67,2025-12-16_NYK_SAS,2025-12-16,NYK,Jalen Brunson,NaN
241,2025-11-08_LAC_PHX,2025-11-08,PHX,Devin Booker,NaN
242,2025-12-10_OKC_PHX,2025-12-10,PHX,Devin Booker,NaN
243,2026-04-08_DAL_PHX,2026-04-08,PHX,Devin Booker,NaN
250,2025-12-16_NYK_SAS,2025-12-16,SAS,Victor Wembanyama,NaN



Absences by player:


,target_games,absences
player,,
Stephen Curry,20,12.0
Jayson Tatum,16,11.0
LeBron James,25,8.0
Anthony Edwards,13,5.0
Shai Gilgeous-Alexander,21,4.0
Luka Doncic,25,3.0
Nikola Jokic,17,2.0
Cade Cunningham,8,2.0
Joel Embiid,7,2.0


In [113]:
#correcting 5 NaN MP values
# Correct the five records where Basketball-Reference did not return an MP value
status_corrections = {
    ("2025-12-16_NYK_SAS", "Jalen Brunson"): 1,
    ("2025-11-08_LAC_PHX", "Devin Booker"): 1,
    ("2025-12-10_OKC_PHX", "Devin Booker"): 0,
    ("2026-04-08_DAL_PHX", "Devin Booker"): 1,
    ("2025-12-16_NYK_SAS", "Victor Wembanyama"): 1,
}

for (match_key, player), played in status_corrections.items():
    mask = (
        (star_availability["match_key"] == match_key) &
        (star_availability["player"] == player)
    )
    
    star_availability.loc[mask, "played"] = played
    star_availability.loc[mask, "absent"] = 1 - played
    star_availability.loc[mask, "source"] = (
        "Basketball-Reference + secondary verification"
    )

# Confirm no unknown statuses remain
print("Unknown statuses:", star_availability["played"].isna().sum())

display(
    star_availability[
        star_availability["match_key"].isin(status_corrections.keys())
    ][["match_key", "player", "mp", "played", "absent", "source"]]
)

Unknown statuses: 0


,match_key,player,mp,played,absent,source


In [114]:
display(
    star_availability[
        star_availability.apply(
            lambda row: (row["match_key"], row["player"]) in status_corrections,
            axis=1
        )
    ][["match_key", "player", "mp", "played", "absent", "source"]]
)

,match_key,player,mp,played,absent,source
67,2025-12-16_NYK_SAS,Jalen Brunson,NaN,1.0,0.0,Basketball-Reference + secondary verification
241,2025-11-08_LAC_PHX,Devin Booker,NaN,1.0,0.0,Basketball-Reference + secondary verification
242,2025-12-10_OKC_PHX,Devin Booker,NaN,0.0,1.0,Basketball-Reference + secondary verification
243,2026-04-08_DAL_PHX,Devin Booker,NaN,1.0,0.0,Basketball-Reference + secondary verification
250,2025-12-16_NYK_SAS,Victor Wembanyama,NaN,1.0,0.0,Basketball-Reference + secondary verification


In [115]:
# Add a provenance note for the five manually verified records
star_availability["notes"] = None

star_availability.loc[
    star_availability["source"] == "Basketball-Reference + secondary verification",
    "notes"
] = "Basketball-Reference row missing; status verified using secondary game evidence"

# Final QA
print("Total records:", len(star_availability))
print("Unique players:", star_availability["player"].nunique())
print("Unique games:", star_availability["match_key"].nunique())
print("Unknown statuses:", star_availability["played"].isna().sum())
print("Total absences:", star_availability["absent"].sum())

print("\nAbsences by player:")
display(
    star_availability
    .groupby("player")
    .agg(
        target_games=("match_key", "count"),
        absences=("absent", "sum")
    )
    .sort_values("absences", ascending=False)
)

Total records: 288
Unique players: 21
Unique games: 127
Unknown statuses: 0
Total absences: 60.0

Absences by player:


,target_games,absences
player,,
Stephen Curry,20,12.0
Jayson Tatum,16,11.0
LeBron James,25,8.0
Anthony Edwards,13,5.0
Shai Gilgeous-Alexander,21,4.0
Luka Doncic,25,3.0
Nikola Jokic,17,2.0
Cade Cunningham,8,2.0
Joel Embiid,7,2.0


In [116]:
#Creating game-level star absence features
# Create one column per star indicating absence
star_absence_wide = (
    star_availability
    .pivot_table(
        index="match_key",
        columns="player",
        values="absent",
        aggfunc="max",
        fill_value=0
    )
    .reset_index()
)

# Rename player columns to clean variable names
star_absence_wide.columns = [
    "match_key"
    if col == "match_key"
    else col.lower().replace(" ", "_").replace("-", "_") + "_absent"
    for col in star_absence_wide.columns
]

# Merge the game date/team information back in
star_absence_wide = target_games.merge(
    star_absence_wide,
    on="match_key",
    how="left"
)

# List of individual absence columns
absence_columns = [
    col for col in star_absence_wide.columns
    if col.endswith("_absent")
]

# Number of designated stars absent
star_absence_wide["star_absence_count"] = (
    star_absence_wide[absence_columns].sum(axis=1)
)

# Number of teams with at least one designated star absent
star_absence_wide["star_absence_team_count"] = (
    (
        (
            (star_absence_wide[absence_columns] == 1)
        )
    )
    .astype(int)
    .sum(axis=1)
)

print("Rows:", len(star_absence_wide))
print("Individual absence variables:", len(absence_columns))

display(
    star_absence_wide[
        [
            "match_key",
            "away_team",
            "home_team",
            "star_absence_count",
            "star_absence_team_count"
        ]
    ].head(15)
)

Rows: 127
Individual absence variables: 21


,match_key,away_team,home_team,star_absence_count,star_absence_team_count
0,2025-10-21_HOU_OKC,HOU,OKC,0,0
1,2025-10-21_GSW_LAL,GSW,LAL,1,1
2,2025-10-22_CLE_NYK,CLE,NYK,0,0
3,2025-10-22_DAL_SAS,SAS,DAL,0,0
4,2025-10-23_IND_OKC,OKC,IND,0,0
5,2025-10-23_DEN_GSW,DEN,GSW,0,0
6,2025-10-24_BOS_NYK,BOS,NYK,1,1
7,2025-10-24_LAL_MIN,MIN,LAL,1,1
8,2025-10-29_LAL_MIN,LAL,MIN,3,3
9,2025-10-29_BOS_CLE,CLE,BOS,1,1


In [117]:
#Correctly calculating number of effected teams
# Map each player to their team
player_to_team = {
    player: team
    for team, players in star_players.items()
    for player in players
}

# Calculate number of affected teams for each game
def count_affected_teams(row):
    absent_players = [
        col.replace("_absent", "")
        for col in absence_columns
        if row[col] == 1
    ]
    
    absent_players = [
        player.replace("_", " ").title()
        for player in absent_players
    ]
    
    affected_teams = {
        player_to_team[player]
        for player in absent_players
        if player in player_to_team
    }
    
    return len(affected_teams)

star_absence_wide["star_absence_team_count"] = (
    star_absence_wide.apply(count_affected_teams, axis=1)
)

display(
    star_absence_wide[
        [
            "match_key",
            "away_team",
            "home_team",
            "star_absence_count",
            "star_absence_team_count"
        ]
        + absence_columns
    ].head(15)
)

,match_key,away_team,home_team,star_absence_count,star_absence_team_count,anthony_edwards_absent,cade_cunningham_absent,cooper_flagg_absent,devin_booker_absent,donovan_mitchell_absent,...,kawhi_leonard_absent,kevin_durant_absent,lebron_james_absent,luka_doncic_absent,nikola_jokic_absent,paolo_banchero_absent,shai_gilgeous_alexander_absent,stephen_curry_absent,tyrese_maxey_absent,victor_wembanyama_absent
0,2025-10-21_HOU_OKC,HOU,OKC,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,2025-10-21_GSW_LAL,GSW,LAL,1,0,0,0,0,0,0,...,0,0,1,0,0,0,0,0,0,0
2,2025-10-22_CLE_NYK,CLE,NYK,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,2025-10-22_DAL_SAS,SAS,DAL,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,2025-10-23_IND_OKC,OKC,IND,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
5,2025-10-23_DEN_GSW,DEN,GSW,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
6,2025-10-24_BOS_NYK,BOS,NYK,1,1,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
7,2025-10-24_LAL_MIN,MIN,LAL,1,0,0,0,0,0,0,...,0,0,1,0,0,0,0,0,0,0
8,2025-10-29_LAL_MIN,LAL,MIN,3,2,1,0,0,0,0,...,0,0,1,1,0,0,0,0,0,0
9,2025-10-29_BOS_CLE,CLE,BOS,1,1,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [118]:
#Inspecting star absence distribution
print("Star absence count distribution:")
display(
    star_absence_wide["star_absence_count"]
    .value_counts()
    .sort_index()
    .rename_axis("star_absence_count")
    .reset_index(name="games")
)

print("\nStar absence team-count distribution:")
display(
    star_absence_wide["star_absence_team_count"]
    .value_counts()
    .sort_index()
    .rename_axis("star_absence_team_count")
    .reset_index(name="games")
)

print("\nGames with at least one star absent:",
      (star_absence_wide["star_absence_count"] > 0).sum())

print("Games with 2+ stars absent:",
      (star_absence_wide["star_absence_count"] >= 2).sum())

print("Maximum star absences in one game:",
      star_absence_wide["star_absence_count"].max())

Star absence count distribution:


,star_absence_count,games
0,0,74
1,1,47
2,2,5
3,3,1



Star absence team-count distribution:


,star_absence_team_count,games
0,0,85
1,1,36
2,2,6



Games with at least one star absent: 53
Games with 2+ stars absent: 6
Maximum star absences in one game: 3


In [119]:
#Games with star absences
# Save the detailed player-level availability data
star_availability_path = (
    raw_path + "star_player_availability_2025_26.csv"
)

star_availability.to_csv(
    star_availability_path,
    index=False
)

# Save the game-level absence features
star_absence_path = (
    raw_path + "star_absence_features_2025_26.csv"
)

star_absence_wide.to_csv(
    star_absence_path,
    index=False
)

# Register the game-level table with DuckDB
con.register("star_absence_wide", star_absence_wide)

# Add the two aggregate absence variables to game_features
con.sql("""
    ALTER TABLE game_features
    ADD COLUMN IF NOT EXISTS star_absence_count INTEGER
""")

con.sql("""
    ALTER TABLE game_features
    ADD COLUMN IF NOT EXISTS star_absence_team_count INTEGER
""")

con.sql("""
    UPDATE game_features AS g
    SET
        star_absence_count = s.star_absence_count,
        star_absence_team_count = s.star_absence_team_count
    FROM star_absence_wide AS s
    WHERE g.match_key = s.match_key
""")

# Verify
verification = con.sql("""
    SELECT
        COUNT(*) AS games,
        COUNT(star_absence_count) AS games_with_absence_data,
        SUM(star_absence_count) AS total_star_absences,
        SUM(CASE WHEN star_absence_count > 0 THEN 1 ELSE 0 END)
            AS games_with_star_absence
    FROM game_features
""").df()

display(verification)

print("Saved:")
print(star_availability_path)
print(star_absence_path)

,games,games_with_absence_data,total_star_absences,games_with_star_absence
0,127,127,60.0,53.0


Saved:
NBA_Global_Media_Insights/Data/raw/star_player_availability_2025_26.csv
NBA_Global_Media_Insights/Data/raw/star_absence_features_2025_26.csv


In [120]:
#Running the audience model with star availability variable
# Build the modeling dataset with star-player absence
model_data_stars = con.sql("""
    SELECT
        audience_millions,
        network,
        high_exposure_team_count,

        CASE
            WHEN away_team = 'BOS' OR home_team = 'BOS'
            THEN 1 ELSE 0
        END AS has_boston,

        is_weekend,
        is_prime_time,
        is_early_game,
        is_opening_night,
        is_nba_cup_other,
        is_nba_cup_final,

        star_absence_count

    FROM game_features
""").df()

# Fit the updated model
star_model = smf.ols(
    """audience_millions ~ C(network)
    + high_exposure_team_count
    + has_boston
    + is_weekend
    + is_prime_time
    + is_early_game
    + is_opening_night
    + is_nba_cup_other
    + is_nba_cup_final
    + star_absence_count""",
    data=model_data_stars
).fit(cov_type="HC3")

print(star_model.summary())

                            OLS Regression Results                            
Dep. Variable:      audience_millions   R-squared:                       0.836
Model:                            OLS   Adj. R-squared:                  0.817
Method:                 Least Squares   F-statistic:                     21.84
Date:                Mon, 28 Sep 2026   Prob (F-statistic):           5.02e-25
Time:                        22:11:38   Log-Likelihood:                -74.959
No. Observations:                 127   AIC:                             177.9
Df Residuals:                     113   BIC:                             217.7
Df Model:                          13                                         
Covariance Type:                  HC3                                         
                                  coef    std err          z      P>|z|      [0.025      0.975]
-----------------------------------------------------------------------------------------------
Intercept         

In [121]:
#Running same model with star absence team count
# Test team-level star absence instead of player-level absence count

model_data_star_teams = con.sql("""
    SELECT
        audience_millions,
        network,
        high_exposure_team_count,

        CASE
            WHEN away_team = 'BOS' OR home_team = 'BOS'
            THEN 1 ELSE 0
        END AS has_boston,

        is_weekend,
        is_prime_time,
        is_early_game,
        is_opening_night,
        is_nba_cup_other,
        is_nba_cup_final,

        star_absence_team_count

    FROM game_features
""").df()

star_team_model = smf.ols(
    """audience_millions ~ C(network)
    + high_exposure_team_count
    + has_boston
    + is_weekend
    + is_prime_time
    + is_early_game
    + is_opening_night
    + is_nba_cup_other
    + is_nba_cup_final
    + star_absence_team_count""",
    data=model_data_star_teams
).fit(cov_type="HC3")

print(star_team_model.summary())

                            OLS Regression Results                            
Dep. Variable:      audience_millions   R-squared:                       0.834
Model:                            OLS   Adj. R-squared:                  0.815
Method:                 Least Squares   F-statistic:                     21.00
Date:                Mon, 28 Sep 2026   Prob (F-statistic):           2.27e-24
Time:                        22:14:07   Log-Likelihood:                -75.537
No. Observations:                 127   AIC:                             179.1
Df Residuals:                     113   BIC:                             218.9
Df Model:                          13                                         
Covariance Type:                  HC3                                         
                                  coef    std err          z      P>|z|      [0.025      0.975]
-----------------------------------------------------------------------------------------------
Intercept         

In [122]:
#Do star absences help explain model residuals (largest misses)?
# Compare model errors with and without star-availability information

# Baseline model: before adding star absences
baseline_data = con.sql("""
    SELECT
        audience_millions,
        network,
        high_exposure_team_count,

        CASE
            WHEN away_team = 'BOS' OR home_team = 'BOS'
            THEN 1 ELSE 0
        END AS has_boston,

        is_weekend,
        is_prime_time,
        is_early_game,
        is_opening_night,
        is_nba_cup_other,
        is_nba_cup_final,

        match_key,
        game_date,
        away_team,
        home_team,
        star_absence_count

    FROM game_features
""").df()

baseline_model = smf.ols(
    """audience_millions ~ C(network)
    + high_exposure_team_count
    + has_boston
    + is_weekend
    + is_prime_time
    + is_early_game
    + is_opening_night
    + is_nba_cup_other
    + is_nba_cup_final""",
    data=baseline_data
).fit(cov_type="HC3")

baseline_data["predicted"] = baseline_model.predict(baseline_data)
baseline_data["residual"] = (
    baseline_data["audience_millions"] - baseline_data["predicted"]
)
baseline_data["abs_error"] = baseline_data["residual"].abs()

# Show the 15 biggest misses and star availability
largest_misses = (
    baseline_data[
        [
            "match_key",
            "game_date",
            "away_team",
            "home_team",
            "audience_millions",
            "predicted",
            "residual",
            "abs_error",
            "star_absence_count"
        ]
    ]
    .sort_values("abs_error", ascending=False)
    .head(15)
)

display(largest_misses)

,match_key,game_date,away_team,home_team,audience_millions,predicted,residual,abs_error,star_absence_count
32,2026-02-22_BOS_LAL,2026-02-22,BOS,LAL,4.522,2.695452,1.826548,1.826548,1
86,2025-12-25_DEN_MIN,2025-12-25,MIN,DEN,3.610,4.930680,-1.320680,1.320680,0
37,2025-12-25_OKC_SAS,2025-12-25,SAS,OKC,6.711,5.737932,0.973068,0.973068,0
76,2025-10-22_DAL_SAS,2025-10-22,SAS,DAL,2.488,1.537009,0.950991,0.950991,0
84,2025-11-26_MIN_OKC,2025-11-26,MIN,OKC,2.433,1.537009,0.895991,0.895991,0
6,2026-03-01_DEN_MIN,2026-03-01,MIN,DEN,2.741,1.861073,0.879927,0.879927,0
1,2025-11-28_MIL_NYK,2025-11-28,MIL,NYK,2.111,1.259802,0.851198,0.851198,0
80,2026-02-01_LAL_NYK,2026-02-01,LAL,NYK,3.601,2.795095,0.805905,0.805905,0
54,2025-11-26_GSW_HOU,2025-11-26,HOU,GSW,2.031,1.253481,0.777519,0.777519,1
117,2025-12-29_DAL_POR,2025-12-29,DAL,POR,1.054,1.798407,-0.744407,0.744407,0


In [124]:
#Missing stars by negative residuals
negative_residuals = (
    baseline_data[
        [
            "match_key",
            "game_date",
            "away_team",
            "home_team",
            "audience_millions",
            "predicted",
            "residual",
            "star_absence_count"
        ]
    ]
    .sort_values("residual")
    .head(15)
)

display(negative_residuals)

,match_key,game_date,away_team,home_team,audience_millions,predicted,residual,star_absence_count
86,2025-12-25_DEN_MIN,2025-12-25,MIN,DEN,3.610,4.930680,-1.320680,0
117,2025-12-29_DAL_POR,2025-12-29,DAL,POR,1.054,1.798407,-0.744407,0
106,2026-01-18_MEM_ORL,2026-01-18,ORL,MEM,0.528,1.269209,-0.741209,0
55,2026-01-19_DAL_NYK,2026-01-19,DAL,NYK,1.866,2.605658,-0.739658,0
66,2026-03-07_GSW_OKC,2026-03-07,GSW,OKC,1.897,2.564826,-0.667826,1
0,2025-10-21_GSW_LAL,2025-10-21,GSW,LAL,3.990,4.633580,-0.643580,1
60,2025-11-08_LAC_PHX,2025-11-08,PHX,LAC,0.516,1.148324,-0.632324,1
74,2026-03-14_LAC_SAC,2026-03-14,SAC,LAC,0.822,1.431851,-0.609851,0
71,2026-01-19_CLE_OKC,2026-01-19,OKC,CLE,1.601,2.196499,-0.595499,0
44,2026-03-15_GSW_NYK,2026-03-15,GSW,NYK,2.225,2.795095,-0.570095,1


In [125]:
#Model Cross Validation between baseline and star absence model
from sklearn.model_selection import KFold, cross_val_score
from sklearn.metrics import mean_squared_error, r2_score
import numpy as np

# ---------------------------------------------------------
# In-sample model comparison
# ---------------------------------------------------------

comparison = pd.DataFrame({
    "metric": [
        "R-squared",
        "Adjusted R-squared",
        "AIC",
        "BIC"
    ],
    "Baseline": [
        baseline_model.rsquared,
        baseline_model.rsquared_adj,
        baseline_model.aic,
        baseline_model.bic
    ],
    "With Star Absence": [
        star_model.rsquared,
        star_model.rsquared_adj,
        star_model.aic,
        star_model.bic
    ]
})

display(comparison)

# ---------------------------------------------------------
# 5-fold cross-validation
# ---------------------------------------------------------

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import Pipeline

X = model_data_stars.drop(columns="audience_millions")
y = model_data_stars["audience_millions"]

categorical_cols = ["network"]
numeric_cols = [
    "high_exposure_team_count",
    "has_boston",
    "is_weekend",
    "is_prime_time",
    "is_early_game",
    "is_opening_night",
    "is_nba_cup_other",
    "is_nba_cup_final",
    "star_absence_count"
]

preprocessor = ColumnTransformer(
    transformers=[
        (
            "network",
            OneHotEncoder(drop="first", handle_unknown="ignore"),
            categorical_cols
        ),
        ("numeric", "passthrough", numeric_cols)
    ]
)

star_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", LinearRegression())
])

cv = KFold(n_splits=5, shuffle=True, random_state=42)

rmse_scores = np.sqrt(
    -cross_val_score(
        star_pipeline,
        X,
        y,
        cv=cv,
        scoring="neg_mean_squared_error"
    )
)

r2_scores = cross_val_score(
    star_pipeline,
    X,
    y,
    cv=cv,
    scoring="r2"
)

print(f"5-fold CV RMSE: {rmse_scores.mean():.3f}M")
print(f"RMSE folds: {[round(x, 3) for x in rmse_scores]}")
print()
print(f"5-fold CV R²: {r2_scores.mean():.3f}")
print(f"R² folds: {[round(x, 3) for x in r2_scores]}")

,metric,Baseline,With Star Absence
0,R-squared,0.831846,0.835695
1,Adjusted R-squared,0.814145,0.816793
2,AIC,178.859314,177.917880
3,BIC,215.833746,217.736499


5-fold CV RMSE: 0.535M
RMSE folds: [0.497, 0.591, 0.388, 0.564, 0.638]

5-fold CV R²: 0.717
R² folds: [0.83, 0.656, 0.817, 0.505, 0.778]


In [126]:
# Build a clean, presentation-ready model results table

model_results = pd.DataFrame({
    "variable": star_model.params.index,
    "coefficient_millions": star_model.params.values,
    "robust_se": star_model.bse.values,
    "p_value": star_model.pvalues.values,
    "ci_lower": star_model.conf_int()[0].values,
    "ci_upper": star_model.conf_int()[1].values
})

# Remove the intercept
model_results = model_results[
    model_results["variable"] != "Intercept"
].copy()

# Create a readable variable name
variable_labels = {
    "C(network)[T.ABC | ESPN]": "ABC | ESPN vs. reference network",
    "C(network)[T.ESPN]": "ESPN vs. reference network",
    "C(network)[T.NBC | Peacock]": "NBC | Peacock vs. reference network",
    "C(network)[T.Prime Video]": "Prime Video vs. reference network",
    "high_exposure_team_count": "High-exposure teams in matchup",
    "has_boston": "Boston matchup",
    "is_weekend": "Weekend",
    "is_prime_time": "Prime-time game",
    "is_early_game": "Early game",
    "is_opening_night": "Opening Night",
    "is_nba_cup_other": "NBA Cup game (non-final)",
    "is_nba_cup_final": "NBA Cup Final",
    "star_absence_count": "Selected star absences"
}

model_results["variable_label"] = (
    model_results["variable"]
    .map(variable_labels)
    .fillna(model_results["variable"])
)

# Convert coefficient / CI into viewer counts
model_results["coefficient_viewers"] = (
    model_results["coefficient_millions"] * 1_000_000
)

model_results["ci_lower_viewers"] = (
    model_results["ci_lower"] * 1_000_000
)

model_results["ci_upper_viewers"] = (
    model_results["ci_upper"] * 1_000_000
)

# Statistical significance flag
model_results["significant_05"] = (
    model_results["p_value"] < 0.05
)

# Reorder columns for readability
model_results = model_results[
    [
        "variable_label",
        "coefficient_millions",
        "robust_se",
        "p_value",
        "ci_lower",
        "ci_upper",
        "significant_05"
    ]
]

# Round for presentation
model_results = model_results.round({
    "coefficient_millions": 3,
    "robust_se": 3,
    "p_value": 3,
    "ci_lower": 3,
    "ci_upper": 3
})

display(model_results)

,variable_label,coefficient_millions,robust_se,p_value,ci_lower,ci_upper,significant_05
1,ABC | ESPN vs. reference network,3.731,0.540,0.000,2.672,4.790,True
2,ESPN vs. reference network,-0.311,0.215,0.149,-0.732,0.111,False
3,NBC | Peacock vs. reference network,0.199,0.217,0.360,-0.226,0.624,False
4,Prime Video vs. reference network,-0.607,0.234,0.009,-1.065,-0.149,True
5,High-exposure teams in matchup,0.426,0.088,0.000,0.252,0.599,True
6,Boston matchup,0.392,0.175,0.025,0.050,0.735,True
7,Weekend,0.326,0.191,0.088,-0.049,0.700,False
8,Prime-time game,0.279,0.168,0.097,-0.051,0.608,False
9,Early game,0.368,0.301,0.221,-0.221,0.958,False
10,Opening Night,2.171,0.874,0.013,0.459,3.884,True


In [127]:
# Final business interpretation of the primary audience model

business_insights = pd.DataFrame([
    {
        "insight": "Selected star absences",
        "finding": "Each additional selected star absence was associated with ~121K fewer viewers",
        "coefficient_millions": star_model.params["star_absence_count"],
        "p_value": star_model.pvalues["star_absence_count"],
        "business_relevance": "Player availability can affect the expected national audience of a matchup."
    },
    {
        "insight": "High-exposure teams",
        "finding": "Each additional high-exposure team was associated with ~426K additional viewers",
        "coefficient_millions": star_model.params["high_exposure_team_count"],
        "p_value": star_model.pvalues["high_exposure_team_count"],
        "business_relevance": "Matchup selection is an important component of national audience potential."
    },
    {
        "insight": "Opening Night",
        "finding": "Opening Night was associated with ~2.17M additional viewers",
        "coefficient_millions": star_model.params["is_opening_night"],
        "p_value": star_model.pvalues["is_opening_night"],
        "business_relevance": "Major league-wide events can create substantial audience premiums."
    },
    {
        "insight": "Boston matchup",
        "finding": "Boston matchups were associated with ~392K additional viewers",
        "coefficient_millions": star_model.params["has_boston"],
        "p_value": star_model.pvalues["has_boston"],
        "business_relevance": "Boston generated additional audience association within this sample after controlling for other factors."
    },
    {
        "insight": "Prime Video",
        "finding": "Prime Video games were associated with ~607K fewer viewers than the reference network",
        "coefficient_millions": star_model.params["C(network)[T.Prime Video]"],
        "p_value": star_model.pvalues["C(network)[T.Prime Video]"],
        "business_relevance": "Observed audience differences vary by network/platform, although network assignment is not random."
    }
])

business_insights["coefficient_millions"] = (
    business_insights["coefficient_millions"].round(3)
)

business_insights["p_value"] = (
    business_insights["p_value"].round(3)
)

display(business_insights)

,insight,finding,coefficient_millions,p_value,business_relevance
0,Selected star absences,Each additional selected star absence was asso...,-0.121,0.033,Player availability can affect the expected na...
1,High-exposure teams,Each additional high-exposure team was associa...,0.426,0.000,Matchup selection is an important component of...
2,Opening Night,Opening Night was associated with ~2.17M addit...,2.171,0.013,Major league-wide events can create substantia...
3,Boston matchup,Boston matchups were associated with ~392K add...,0.392,0.025,Boston generated additional audience associati...
4,Prime Video,Prime Video games were associated with ~607K f...,-0.607,0.009,Observed audience differences vary by network/...


In [128]:
#final table for Tableau
# Create the final Tableau-ready game-level analytical table

tableau_game_data = con.sql("""
    SELECT
        match_key,
        game_date,
        away_team,
        home_team,

        network,
        season_type,

        tip_et,
        day_of_week,
        tip_hour,

        audience_millions,

        -- Core matchup factors
        high_exposure_team_count,
        has_lal,
        has_nyk,
        has_okc,
        has_gsw,
        has_dal,

        -- Timing
        is_weekend,
        is_prime_time,
        is_early_game,

        -- Event context
        is_opening_night,
        is_christmas,
        is_nba_cup_other,
        is_nba_cup_final,

        -- Player availability
        star_absence_count,
        star_absence_team_count

    FROM game_features
    ORDER BY game_date
""").df()

print("Shape:", tableau_game_data.shape)
print()
print("Columns:")
print(tableau_game_data.columns.tolist())
print()

display(tableau_game_data.head())

Shape: (127, 25)

Columns:
['match_key', 'game_date', 'away_team', 'home_team', 'network', 'season_type', 'tip_et', 'day_of_week', 'tip_hour', 'audience_millions', 'high_exposure_team_count', 'has_lal', 'has_nyk', 'has_okc', 'has_gsw', 'has_dal', 'is_weekend', 'is_prime_time', 'is_early_game', 'is_opening_night', 'is_christmas', 'is_nba_cup_other', 'is_nba_cup_final', 'star_absence_count', 'star_absence_team_count']



,match_key,game_date,away_team,home_team,network,season_type,tip_et,day_of_week,tip_hour,audience_millions,...,has_dal,is_weekend,is_prime_time,is_early_game,is_opening_night,is_christmas,is_nba_cup_other,is_nba_cup_final,star_absence_count,star_absence_team_count
0,2025-10-21_HOU_OKC,2025-10-21,HOU,OKC,NBC | Peacock,Regular Season,2025-10-21T18:30:00-05:00,Tuesday,18,4.868,...,0,0,1,0,1,0,0,0,0,0
1,2025-10-21_GSW_LAL,2025-10-21,GSW,LAL,NBC | Peacock,Regular Season,2025-10-21T21:00:00-05:00,Tuesday,21,3.990,...,0,0,1,0,1,0,0,0,1,0
2,2025-10-22_CLE_NYK,2025-10-22,CLE,NYK,ESPN,Regular Season,2025-10-22T18:00:00-05:00,Wednesday,18,2.166,...,0,0,1,0,0,0,0,0,0,0
3,2025-10-22_DAL_SAS,2025-10-22,SAS,DAL,ESPN,Regular Season,2025-10-22T20:30:00-05:00,Wednesday,20,2.488,...,1,0,1,0,0,0,0,0,0,0
4,2025-10-23_IND_OKC,2025-10-23,OKC,IND,ESPN,Regular Season,2025-10-23T18:30:00-05:00,Thursday,18,1.975,...,0,0,1,0,0,0,0,0,0,0


In [129]:
# Save final analytical outputs for Tableau and project documentation

tableau_game_path = (
    raw_path + "tableau_game_data_2025_26.csv"
)

model_results_path = (
    raw_path + "model_results_2025_26.csv"
)

business_insights_path = (
    raw_path + "business_insights_2025_26.csv"
)

tableau_game_data.to_csv(
    tableau_game_path,
    index=False
)

model_results.to_csv(
    model_results_path,
    index=False
)

business_insights.to_csv(
    business_insights_path,
    index=False
)

print("Saved:")
print(tableau_game_path)
print(model_results_path)
print(business_insights_path)

Saved:
NBA_Global_Media_Insights/Data/raw/tableau_game_data_2025_26.csv
NBA_Global_Media_Insights/Data/raw/model_results_2025_26.csv
NBA_Global_Media_Insights/Data/raw/business_insights_2025_26.csv


In [130]:
prediction_check = con.sql("""
    SELECT
        match_key,
        game_date,
        away_team,
        home_team,
        audience_millions,
        network,
        star_absence_count,
        high_exposure_team_count
    FROM game_features
""").df()

prediction_check["predicted_audience_millions"] = star_model.predict(
    model_data_stars
)

prediction_check["residual_millions"] = (
    prediction_check["audience_millions"]
    - prediction_check["predicted_audience_millions"]
)

prediction_check.to_csv(
    raw_path + "tableau_prediction_data_2025_26.csv",
    index=False
)

print(prediction_check.shape)
print(prediction_check.head())

(127, 10)
            match_key  game_date away_team home_team  audience_millions  \
0  2025-10-21_GSW_LAL 2025-10-21       GSW       LAL              3.990   
1  2025-11-28_MIL_NYK 2025-11-28       MIL       NYK              2.111   
2  2026-02-03_CHI_MIL 2026-02-03       CHI       MIL              1.050   
3  2025-11-07_DEN_GSW 2025-11-07       GSW       DEN              1.060   
4  2025-11-14_GSW_SAS 2025-11-14       GSW       SAS              1.434   

         network  star_absence_count  high_exposure_team_count  \
0  NBC | Peacock                   1                         2   
1    Prime Video                   0                         1   
2           ESPN                   1                         0   
3    Prime Video                   1                         1   
4    Prime Video                   0                         1   

   predicted_audience_millions  residual_millions  
0                     4.581549          -0.591549  
1                     1.299624        